In [ ]:
pip install curl_cffi

In [ ]:
import json, html, re, pathlib
from curl_cffi import requests as creq

# GAME_ID 는 위 EVENT 에 속한 경기여야 한다. 다른 대회의 번호를 넣으면
# 404 가 아니라 200 에 일반 페이지가 와서 "data-page 없음" 으로 죽는다.
GAME_ID = 207377
# 대회 주소 — 호스트가 대회마다 다르다. WBSC 본부가 여는 월드컵은 wbsc.org 에,
# 아시아(BFA) 대륙 대회는 wbscasia.org 에 올라온다. 페이지 구조(data-page JSON)는
# 두 곳이 같아서 아래 두 줄만 바꾸면 파서는 그대로 돈다.
#   2025 U-18 월드컵 : HOST="https://www.wbsc.org"      EVENT="2025-u18-baseball-world-cup"
#   2026 BFA U-18    : HOST="https://www.wbscasia.org"  EVENT="2026-bfa-xiv-u18-championship"
HOST  = "https://www.wbscasia.org"
EVENT = "2026-bfa-xiv-u18-championship"
URL = f"{HOST}/en/events/{EVENT}/schedule-and-results/box-score/{GAME_ID}"

OUT = pathlib.Path("wbsc"); OUT.mkdir(exist_ok=True)

r = creq.get(URL, impersonate="chrome", timeout=30)
print("status:", r.status_code, "| len:", len(r.text))

m = re.search(r'data-page="([^"]+)"', r.text)
if not m:
    (OUT / "debug.html").write_text(r.text, encoding="utf-8")
    raise SystemExit("data-page 없음 — debug.html 확인")

data = json.loads(html.unescape(m.group(1)))
(OUT / f"{GAME_ID}.json").write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

print("props 키:", list(data.get("props", {}).keys()))

In [ ]:
orig = data["props"]["viewData"]["original"]
print(type(orig))
print(list(orig.keys()) if isinstance(orig, dict) else orig[:200])

In [ ]:
orig = data["props"]["viewData"]["original"]
gp = orig["gamePlays"]

print("type:", type(gp).__name__)
print("len:", len(gp))
print()

if isinstance(gp, dict):
    print("keys:", list(gp.keys()))
    first = next(iter(gp.values()))
elif isinstance(gp, list):
    first = gp[0]

print("\n=== 첫 원소 ===")
import json
print(json.dumps(first, ensure_ascii=False, indent=2)[:2500])

In [ ]:
"""
WBSC gamePlays 파서

경로: props.viewData.original.gamePlays.all[inning][top|bot] -> list of rows
레코드는 투구/이벤트 단위. PA 단위는 pa==1 로 마감된다(추정 -> 아래에서 검증).
"""

import json, html, re, pathlib
import pandas as pd
from curl_cffi import requests as creq

HOST  = "https://www.wbscasia.org"        # 월드컵은 "https://www.wbsc.org"
EVENT = "2026-bfa-xiv-u18-championship"   # 월드컵은 "2025-u18-baseball-world-cup"
OUT = pathlib.Path("wbsc"); OUT.mkdir(exist_ok=True)


def fetch_game(game_id: int) -> dict:
    url = f"{HOST}/en/events/{EVENT}/schedule-and-results/box-score/{game_id}"
    r = creq.get(url, impersonate="chrome", timeout=30)
    r.raise_for_status()
    m = re.search(r'data-page="([^"]+)"', r.text)
    if not m:
        raise RuntimeError(f"{game_id}: data-page 없음")
    return json.loads(html.unescape(m.group(1)))["props"]["viewData"]["original"]


def plays_to_df(orig: dict) -> pd.DataFrame:
    """gamePlays.all 을 평탄화."""
    rows = []
    for inning, halves in orig["gamePlays"]["all"].items():
        for half, plays in halves.items():          # 'top' / 'bot'
            for p in plays:
                p = dict(p)
                p["_inning"] = int(inning)
                p["_half"] = half
                rows.append(p)

    df = pd.DataFrame(rows)
    # 원본에 playorder 가 있으니 그걸로 정렬 (없으면 등장 순서)
    sort_cols = [c for c in ("_inning", "_half", "playorder") if c in df.columns]
    df = df.sort_values(sort_cols, kind="stable").reset_index(drop=True)
    return df


def add_players(df: pd.DataFrame, orig: dict) -> pd.DataFrame:
    """batterid / pitcherid -> 이름 매핑 (boxScore 에서 뽑는다)."""
    name = {}

    def harvest(obj):
        if isinstance(obj, dict):
            pid = obj.get("playerid") or obj.get("id")
            nm = obj.get("name") or obj.get("fullname") or obj.get("player")
            if not nm:
                # boxScore 의 선수 객체는 name 을 주지 않고 lastname/firstname 으로
                # 쪼개 준다. 이걸 안 봐서 이 셀은 여태 이름을 한 명도 못 붙였다
                # (2025 대회 출력에도 "매핑된 선수 0명" 으로 남아 있다).
                # 실제 파이프라인은 셀 6 의 build_player_lookup 을 쓰므로 산출물은
                # 멀쩡했고, 여기 눈으로 보는 CSV 만 비어 있었다.
                ln, fn = obj.get("lastname"), obj.get("firstname")
                if isinstance(ln, str) or isinstance(fn, str):
                    nm = f"{ln or ''} {fn or ''}".strip()
            if isinstance(pid, int) and isinstance(nm, str) and nm:
                name.setdefault(pid, nm)
            for v in obj.values():
                harvest(v)
        elif isinstance(obj, list):
            for v in obj:
                harvest(v)

    harvest(orig.get("boxScore"))
    df["batter"] = df["batterid"].map(name)
    df["pitcher"] = df["pitcherid"].map(name)
    print(f"[name] 매핑된 선수 {len(name)}명 / "
          f"batter 결측 {df['batter'].isna().sum()}, pitcher 결측 {df['pitcher'].isna().sum()}")
    return df


def inspect(df: pd.DataFrame):
    """필드 의미 검증용 — 값 분포 찍기."""
    print(f"\n총 레코드: {len(df):,}")
    print(f"이닝: {sorted(df['_inning'].unique())}")

    for c in ["nopitch", "pa", "ab", "outs", "runner1", "runner2", "runner3",
              "runs", "out", "inplay"]:
        if c in df.columns:
            vc = df[c].value_counts().sort_index().to_dict()
            print(f"  {c:10s} {vc}")

    print("\n=== PA 마감 레코드 (pa==1) 샘플 ===")
    pa = df[df["pa"] == 1]
    print(f"PA 총 {len(pa)}개")
    cols = ["_inning", "_half", "outs", "runner1", "runner2", "runner3",
            "batter", "pitcher", "runs", "out", "narrative"]
    cols = [c for c in cols if c in pa.columns]
    print(pa[cols].head(15).to_string())

    print("\n=== narrative 있는 레코드 (텍스트 로그) ===")
    nar = df[df["narrative"].astype(str).str.strip() != ""]
    print(f"{len(nar)}개")
    print(nar[["_inning", "_half", "outs", "narrative"]].head(20).to_string())


if __name__ == "__main__":
    GAME_ID = 207380   # 2026 BFA 한국-대만 (EVENT 안의 경기여야 한다)

    orig = fetch_game(GAME_ID)
    df = plays_to_df(orig)
    df = add_players(df, orig)

    df.to_csv(OUT / f"{GAME_ID}_plays.csv", index=False, encoding="utf-8-sig")
    print(f"저장: {OUT / f'{GAME_ID}_plays.csv'}")

    inspect(df)

In [ ]:
for c in ["nopitch","pa","ab","outs","runner1","runner2","runner3","runs","out"]:
    print(c, df[c].value_counts().sort_index().to_dict())
print("총", len(df))

In [ ]:
"""
WBSC gamePlays -> KBSA PBP 포맷 변환기

KBSA_크롤링 노트북의 extract_game_data() 최종 컬럼 구조에 맞춘다. KBSA와 달리
on_base_123 / away_score / home_score 는 여기서 **직접 채운다** — KBSA가 그 열을
비워두고 "3. 주자상황 체크"에서 시뮬레이션으로 복원하는 이유는 종이 기록지에
주자 정보가 없기 때문인데, WBSC는 runner1~3 / runner1after~3after / runs 로
정답을 그대로 주기 때문이다. (예외_아웃만 KBSA와 동일하게 빈 칸으로 남긴다.)

이 변환기가 만드는 행 종류:
  1. 타석 완료 행 (pa==1)              — result = classify_result()의 한글 코드
  2. 주루 이벤트 후속행 (batter 없음)  — 도루/도루자/견제사/폭투/포일/보크/실책/주자아웃
  3. 대주자 행                         — Pinch Runner 교체
  4. 승부주자 행                       — WBSC Tie Break 연장 주자 배치
2~4번이 없으면 KBSA 파이프라인이 깨진다:
  - 도루자/견제사가 없으면 half의 아웃이 3개가 안 채워지고(50경기 중 28경기에서
    총 42아웃 누락), 잡힌 주자가 계속 루상에 남아 주자상황이 어긋난다.
  - 대주자가 없으면 compare_batter_box의 _player_map이 타순->원래 선수로 남아
    이후 득점(R)이 교체된 선수에게 잘못 붙는다.
  - 승부치기 주자가 없으면 연장 이닝이 주자 0명으로 시작한다(8경기 16개 half).

outs_when_up 은 "타석에 들어설 때 누적 아웃수"가 아니라 KBSA calculate_outs()와
같은 의미다 — "이 행이 만든 아웃 개수". 한 플레이에서 타자와 별개로 기존 주자가
아웃되면(안타 뒤 주루사, 야수선택, 병살 등) 그 몫은 '주자아웃' 후속행으로 분리한다.

result 열은 다음 셀의 classify_result()로 번역한 값이고, 원문(result_en)과
번역 신뢰도(result_confidence: high/med/low/none)를 뒤에 추가로 남긴다.

행 순서: 기존 KBSA 엑셀과 동일하게 원정팀(top) 1~n회 전체 -> 홈팀(bot) 1~n회
전체 순이다. plate_seq(팀별 타석 순번)는 실제 시간순(playorder)으로 먼저 매긴 뒤
표시 순서만 바꾼다.

주의: convert_game()은 classify_result(), apply_korea_name_map()(다음 셀들에서
정의)을 호출한다. 실제 실행/저장은 그 셀들이 모두 정의된 뒤 맨 마지막 셀에서 한다.
"""

import json, html, re, pathlib
import pandas as pd
from curl_cffi import requests as creq

HOST  = "https://www.wbscasia.org"        # 월드컵은 "https://www.wbsc.org"
EVENT = "2026-bfa-xiv-u18-championship"   # 월드컵은 "2025-u18-baseball-world-cup"
OUT = pathlib.Path("wbsc"); OUT.mkdir(exist_ok=True)

KBSA_COLUMNS = [
    'game_id', 'team', 'plate_seq', 'inning', 'topbot', 'order', '예외_아웃', 'batter',
    'on_base_123', 'result', 'outs_when_up', 'away_score', 'home_score', 'RBI', 'position',
    'tournament', 'details', 'year', 'date', 'time', 'stadium',
    'team1', 'score1', 'result1', 'team2', 'score2', 'result2',
    'def_team', 'pitcher'
]
EXTRA_REVIEW_COLUMNS = ['result_en', 'result_confidence']
# 주자 상황을 눈으로 보고 검산할 수 있게 내는 열. KBSA "3. 주자상황 체크" 가
# 시뮬레이션으로 복원해 _주자상황.xlsx 에 싣는 것과 같은 짜임이다 — 다만 여기서는
# 추정하지 않는다. WBSC 가 runner1~3 / runner1~3after / runner1~3out / runnerNto4 로
# 정답을 주므로 그대로 옮기기만 하면 된다.
RUNNER_COLUMNS = ['outs_before', 'base_1_run', 'base_2_run', 'base_3_run',
                  'scored_run', 'out_runner']


def fetch_game(game_id: int) -> dict:
    url = f"{HOST}/en/events/{EVENT}/schedule-and-results/box-score/{game_id}"
    r = creq.get(url, impersonate="chrome", timeout=30)
    r.raise_for_status()
    m = re.search(r'data-page="([^"]+)"', r.text)
    if not m:
        raise RuntimeError(f"{game_id}: data-page 없음")
    return json.loads(html.unescape(m.group(1)))["props"]["viewData"]["original"]


# ── 배너/교체 행 판별 ────────────────────────────────────────────────
_BANNER_RE = re.compile(r'^(Middle of the|End of the|GAME OVER|Play Ball|WBSC Tie Break)')
# 교체/공지 행은 runner1~3 / lineupslot 이 전부 0으로 비어 있어 주자상황에 쓸 수 없다
_SUB_RE = re.compile(r'^(Pinch |Defensive |Offensive |Pitching Change)')
_PITCH_CHANGE_RE = re.compile(r'^Pitching Change:\s*#(\S+)\s+(.*?)\s+for\s+#(\S+)\s', re.S)
# "Pinch Runner: #17 SAEZ MEDINA for #39 BORRERO FELICANO at 2nd"
_PINCH_RUN_RE = re.compile(r'^Pinch Runner:\s*#(\S+)\s', re.S)


def is_banner(narrative) -> bool:
    """이닝 구분/경기종료/승부치기 안내 같은 '플레이가 아닌' 배너 행인지.
    주의: 'MOSER Jules  out by rule:<br>Dropped Infield Fly' 처럼 실제 플레이에도
    <br>이 들어가므로 '<br> 포함' 만으로 판정하면 안 된다."""
    return bool(_BANNER_RE.match(str(narrative or '').strip()))


def is_tiebreak(narrative) -> bool:
    return str(narrative or '').strip().startswith('WBSC Tie Break')


# ── 주자/타순 유틸 ───────────────────────────────────────────────────
def slot_order(slot) -> int:
    """runner1 / runner1after 등에 들어있는 lineupslot 코드에서 타순을 뽑는다.
    형식: <1=away|2=home><타순 2자리><그 타순 안에서의 교체 순번>
    (예: 2050 = 홈 5번타순 원선수, 1051 = 원정 5번타순 첫 교체선수). 0/None은 주자 없음."""
    try:
        s = int(slot)
    except (TypeError, ValueError):
        return 0
    return (s // 10) % 100 if s else 0


def bases_before(row) -> dict:
    return {b: row.get(f'runner{b}') for b in (1, 2, 3) if row.get(f'runner{b}')}


def bases_after(row) -> dict:
    return {b: row.get(f'runner{b}after') for b in (1, 2, 3) if row.get(f'runner{b}after')}


def fmt_bases(d) -> object:
    """{1,3} -> '13' / 빈 루상 -> 0 (KBSA on_base_123 표기)."""
    return ''.join(str(b) for b in sorted(d)) if d else 0


def runner_cells(bases, slot_map, name_map) -> dict:
    """{루: lineupslot} -> {'base_1_run': 이름, ...}. 빈 루는 None."""
    out = {f'base_{b}_run': None for b in (1, 2, 3)}
    for b, slot in bases.items():
        out[f'base_{b}_run'] = name_map.get(slot_map.get(slot))
    return out


def scored_names(row, before, slot_map, name_map, batter_name) -> object:
    """이 플레이에서 홈을 밟은 주자 이름.

    runnerNto4 가 "N루에서 홈까지 갔다" 는 뜻이고 runner0 은 타자다. 득점 수(runs)로는
    누가 들어왔는지 알 수 없어서 이 열이 따로 필요하다 — 승계주자 책임이나 타점
    귀속을 눈으로 확인할 때 결국 이걸 봐야 한다."""
    names = []
    if row.get("runner0to4") and batter_name:
        names.append(batter_name)
    for b in (1, 2, 3):
        if row.get(f"runner{b}to4") and b in before:
            nm = name_map.get(slot_map.get(before[b]))
            if nm:
                names.append(nm)
    return ", ".join(names) or None


def out_runner_names(row, before, slot_map, name_map) -> object:
    """이 플레이에서 아웃된 '기존 주자' 이름 (몇 루에 있던 주자인지까지).

    타자 아웃은 넣지 않는다 — 그건 result 코드가 이미 말해 준다. 야수선택(2땅R)이나
    병살에서 정작 알고 싶은 건 "누가 죽었나" 인데, 아웃 수만으로는 알 수 없다."""
    got = []
    for b in (1, 2, 3):
        if row.get(f"runner{b}out") and b in before:
            nm = name_map.get(slot_map.get(before[b]))
            got.append(f"{b}루 {nm}" if nm else f"{b}루")
    return ", ".join(got) or None


def outs_produced(row) -> int:
    """이 플레이(타석 완료든 도루실패/견제사 같은 비타석 이벤트든)가 만든 아웃 수.

    WBSC narrative는 아웃이 발생한 플레이 끝에 "이닝 누적 아웃수 + out." 을 붙이는
    표기 규칙을 쓴다(예: "grounds out 6-3. 2 out.", 병살이면 한 번에 2계단 뛴다).
    그 숫자와 이 행의 outs(그 플레이 전까지의 누적 아웃수) 차이를 보면 되므로,
    gdp/strikeout 같은 플래그를 일일이 나열하는 것보다 훨씬 안정적이다.
    (예전에는 플래그 기반으로 셌는데, 도루실패/견제사처럼 pa==0인 아웃을 놓치고,
    "reaches on fielders choice"인데 실제로는 아무도 안 죽은 케이스를 아웃으로
    잘못 세는 문제가 있었다.)"""
    nar = str(row.get('narrative', '') or '')
    before = row.get('outs') or 0
    m = re.search(r'(\d+)\s+out\.', nar)
    if m:
        return max(0, int(m.group(1)) - before)
    # WBSC 가 긴 원문을 잘라 "out." 이 통째로 날아가고 숫자만 남는 경우가 있다
    # (2026 207393 6회 희생플라이: "… to 2nd on throw. 2"). 끝에 남은 숫자가
    # 이닝 누적 아웃수로 말이 될 때만(지금 아웃수보다 크고 3 이하) 그것으로 본다 —
    # "out at 2nd 6" 같은 수비 위치 번호를 아웃수로 잘못 읽지 않기 위해서다.
    m = re.search(r'(\d+)\s*$', nar)
    if m:
        n = int(m.group(1))
        if before < n <= 3:
            return n - before
    return 0


_outs_produced = outs_produced   # 예전 이름 호환


def plays_to_df(orig: dict) -> pd.DataFrame:
    """gamePlays.all 을 평탄화 (투구 단위, PA는 pa==1 로 마감)."""
    rows = []
    for inning, halves in orig["gamePlays"]["all"].items():
        for half, plays in halves.items():
            for p in plays:
                p = dict(p)
                p["_inning"] = int(inning)
                p["_half"] = half
                rows.append(p)
    df = pd.DataFrame(rows)
    # _half는 문자열이라 그대로 정렬하면 "bot"(b)이 "top"(t)보다 앞에 와서 초/말이
    # 뒤집힌다. playorder는 이닝 전체에 걸쳐 전역적으로 단조 증가하므로 이걸 쓴다.
    sort_cols = [c for c in ("_inning", "playorder") if c in df.columns]
    return df.sort_values(sort_cols, kind="stable").reset_index(drop=True)


# ── boxScore 기반 매핑 3종 ───────────────────────────────────────────
def _iter_box_players(orig: dict):
    """boxScore를 순회하며 (side, 타순, 그 타순 안 인덱스, 선수dict) 를 내놓는다."""
    gd = orig["gameData"]
    bs = orig.get("boxScore", {})
    if not isinstance(bs, dict):
        return
    for tid, val in bs.items():
        if tid in ("totals", "pitchers") or not isinstance(val, dict):
            continue
        try:
            t = int(tid)
        except (TypeError, ValueError):
            continue
        side = 'away' if t == gd.get("awayid") else ('home' if t == gd.get("homeid") else None)
        if side is None:
            continue
        for spot, players in val.items():
            if not isinstance(players, list):
                continue
            for i, p in enumerate(players):
                if p.get("playerid") is not None:
                    yield side, int(spot), i, p


def build_player_lookup(orig: dict):
    """playerid -> (이름, 포지션)."""
    name_map, pos_map = {}, {}
    for _side, _spot, _i, p in _iter_box_players(orig):
        full = f"{p.get('lastname', '')} {p.get('firstname', '')}".strip()
        if full:
            name_map.setdefault(p["playerid"], full)
        if p.get("pos"):
            pos_map.setdefault(p["playerid"], p["pos"])
    return name_map, pos_map


def build_slot_map(orig: dict) -> dict:
    """lineupslot 코드 -> playerid.

    boxScore[teamid][타순] 리스트의 인덱스가 곧 lineupslot의 마지막 자리(교체 순번)다.

    이게 중요한 이유: WBSC의 batterid는 같은 타순 교체 직후 한 타석에서 이전 선수
    값으로 남아있는 경우가 있는데(narrative/boxScore와 불일치 — 2025 대회 50경기에서
    2건), lineupslot은 전수 확인 결과 100% 정확하다. 그래서 타자 식별은 batterid가
    아니라 lineupslot으로 한다."""
    out = {}
    for side, spot, i, p in _iter_box_players(orig):
        out[(1 if side == 'away' else 2) * 1000 + spot * 10 + i] = p["playerid"]
    return out


def build_uniform_map(orig: dict) -> dict:
    """(side, 등번호) -> (playerid, 타순)."""
    return {(side, str(p.get("uniform"))): (p["playerid"], spot)
            for side, spot, _i, p in _iter_box_players(orig)}


def build_pitcher_timeline(orig: dict, df: pd.DataFrame) -> list:
    """행별 '현재 투수' playerid 리스트 (df와 같은 길이).

    WBSC의 pitcherslot은 항상 0으로 비어있고, pitcherid는 batterid와 같은 stale
    문제가 있다(2025 대회 game 173037: 교체 후 3타석이 이전 투수로 남아 공식 BF
    9/3이 12/0으로 집계됐다). 그래서 선발(pitch_gs)에서 시작해
    "Pitching Change: #14 QUIGG for #10 KENNEDY" 행을 만날 때마다 등번호로
    갈아끼우는 방식으로 직접 추적한다."""
    uni2pid, starter = {}, {}
    for side, _spot, _i, p in _iter_box_players(orig):
        uni2pid[(side, str(p.get("uniform")))] = p["playerid"]
        if p.get("pitch_gs"):
            starter[side] = p["playerid"]
    cur = dict(starter)
    for side, half in (('home', 'top'), ('away', 'bot')):   # 수비(투구)하는 쪽
        if side not in cur:
            s = df[(df["_half"] == half) & df["pitcherid"].notna()]
            if not s.empty:
                cur[side] = s.iloc[0]["pitcherid"]

    timeline = []
    for _, row in df.iterrows():
        side = 'home' if row["_half"] == 'top' else 'away'
        m = _PITCH_CHANGE_RE.match(str(row.get("narrative", "") or "").strip())
        if m:
            pid = uni2pid.get((side, m.group(1)))
            if pid is not None:
                cur[side] = pid
        timeline.append(cur.get(side))
    return timeline


# ── 주자 아웃 분리 / 비타석 이벤트 코드 ──────────────────────────────
_OUT_AT_BASE = {'1st': 1, '2nd': 2, '3rd': 3, 'home': 3}


def split_runner_outs(row, after, slot_map, raw_name, narrative):
    """이 타석에서 아웃된 '기존 주자'들을 [(가상 점유 루, lineupslot), ...] 로 돌려준다.

    PA 행의 on_base_123은 '플레이 후' 상태라 아웃된 주자가 이미 빠져 있다.
    step5_track에 "이 주자가 아웃됐다"고 알려주려면 아웃 직전 상태(= 아웃된 주자가
    아직 어딘가 서 있는 상태)를 on_base_123에 넣고 곧바로 '주자아웃' 후속행으로
    빼주면 된다. 그 '어딘가'는 narrative의 "out at 2nd" 표기에서 가져온다
    (홈에서 아웃이면 3루에 있었던 것으로 본다).

    분리하지 않으면 step5_track은 "살아남은 주자 수 > 루상 슬롯 수" 상황에서
    누가 아웃됐는지 알 수 없어 리드주자를 남기고 타자를 버리는 식으로 주자 정체를
    뒤바꾼다(그러면 이후 득점 R이 엉뚱한 선수에게 붙는다).

    아웃 주자 전원에게 빈 루를 배정할 수 없으면 분리를 포기한다 — 반쪽만 분리하면
    아웃 수가 어긋나기 때문이다."""
    before = bases_before(row)
    out_bases = [b for b in (3, 2, 1) if row.get(f'runner{b}out') and b in before]
    if not out_bases:
        return []
    placed, used = [], set(after)
    for b in out_bases:
        nm = raw_name.get(slot_map.get(before[b]))
        m = re.search(re.escape(nm) + r'\s+out at (\w+)', narrative) if nm else None
        want = _OUT_AT_BASE.get(m.group(1), b) if m else b
        while want in used and want > 1:
            want -= 1
        if want in used:
            return []
        used.add(want)
        placed.append((want, before[b]))
    return placed


def _minus_names(names, drop):
    """'A, B' 꼴 이름 목록에서 drop 에 든 이름을 뺀다. 남는 게 없으면 None."""
    if not names or not drop:
        return names
    kept = [n for n in str(names).split(", ") if n not in drop]
    return ", ".join(kept) or None


def batter_out_on_bases(row, narrative, bid, raw_name, taken):
    """타자 본인이 출루한 뒤 루상에서 잡힌 경우 (출루한 루, lineupslot). 아니면 None.

    예: "JUNSANG EOM singles through the left side. JUNSANG EOM out at 2nd 74."
    안타로 1루에 먼저 섰다가 더 가다 죽은 것이라, 타석 행은 그 루에 세워 두고
    '주자아웃' 후속행에서 뺀다. WBSC 는 이걸 runner0out 으로 표시하지 않아
    (2026 전 경기 0건) 원문에서 타자 이름 + "out at" 을 찾는다."""
    if row.get('homerun') == 1:
        return None
    if not (row.get('h') == 1 or 'reaches on' in narrative.lower()):
        return None
    nm = raw_name.get(bid)
    if not nm or not re.search(re.escape(str(nm)) + r'\s+out at', narrative):
        return None
    base = 3 if row.get('triple') == 1 else 2 if row.get('double') == 1 else 1
    if base in taken:
        return None
    return base, row.get('lineupslot')


def _group_moves(moves):
    """[(사유, 이름, 도착루), ...] -> [(사유, [(이름, 도착루), ...]), ...]

    같은 투구의 같은 사유는 한 사건으로 묶는다. 도루 때 포수 송구가 빠지면 3루 주자
    득점과 도루 주자의 추가 진루가 한 번의 실책으로 함께 일어나는데, 원문은 그걸
    "A scores on E2T. B to 2nd on stolen base. B to 3rd on E2T." 처럼 흩어 적는다
    (208117 2회). 원문 순서대로 끊으면 실책이 두 번 난 것처럼 된다.
    도루는 맨 앞에 둔다 — 실책·폭투는 도루 시도 중이나 그 뒤에 나오므로, 뒤에 두지
    않으면 도루 주자가 3루까지 갔다가 2루로 되돌아가는 꼴이 된다("2루 도루 후
    폭투로 3루" 도 같은 순서다)."""
    codes = sorted(dict.fromkeys(c for c, _, _ in moves), key=lambda c: c != '도루')
    return [(c, [(w, d) for cc, w, d in moves if cc == c]) for c in codes]


def _apply_group(state, mv, slot_of):
    """묶인 진루들을 상태에 반영한다. 앞선 주자부터 움직여야 3루 주자가 홈에 들기
    전에 뒤 주자가 3루를 덮어쓰지 않는다. (주자순서 문자열, 득점수, 득점자 slot들)
    을 돌려주고, 주자를 못 찾으면 None."""
    def cur_base(who):
        slot = slot_of.get(who)
        return next((b for b, s in state.items() if s == slot), 0)

    orders, runs, scorers = [], 0, []
    for who, dest in sorted(mv, key=lambda wd: -cur_base(wd[0])):
        slot = slot_of.get(who)
        if slot is None:
            return None
        for b, s in list(state.items()):
            if s == slot:
                del state[b]
        if dest == 4:
            runs += 1
            scorers.append(slot)
        else:
            state[dest] = slot
        orders.append(str(slot_order(slot)))
    return "".join(dict.fromkeys(orders)), runs, scorers


_CS_RE = re.compile(r'\bCS\d')
_PO_RE = re.compile(r'\bPO\d')
_ERR_RE = re.compile(r'\bE\d')
# 실책 표기는 E<자리> 가 포구, E<자리>T 가 송구다. KBSA 는 주루 이벤트 행에
# 자리번호를 적지 않고 종류만 가른다(송구실책 5,933건 / 포구실책 1,447건,
# 뭉뚱그린 '실책' 은 15건뿐이라 사실상 쓰지 않는다).
_THROW_ERR_RE = re.compile(r'\bE(\d)T\b')


def error_code(text) -> str:
    """실책 진루의 결과 코드.

    KBSA 는 '송구실책' 한 마디로 끝내지만 WBSC 는 E1T 처럼 자리번호를 준다.
    국가대표 자료에서는 그걸 살려 '투송실'(투수 송구실책)처럼 누가 저질렀는지까지
    남긴다 — KBSA 어휘 밖이라 검수에서는 허용목록으로 통과시킨다.
    자리번호가 없으면(원문이 말로만 실책을 밝힌 경우) KBSA 표기로 물러선다.
    포구 실책도 짝을 맞춰 '{자리}포실'(포수면 '포포실')로 적는다.
    """
    t = str(text or '')
    m = _THROW_ERR_RE.search(t)
    if m:
        pos = std_pos(m.group(1))
        return f'{pos}송실' if pos else '송구실책'
    if 'throwing error' in t.lower():
        return '송구실책'
    m = _ERR_RE.search(t)
    pos = std_pos(m.group(0)[1:]) if m else None
    return f'{pos}포실' if pos else '포구실책'


def event_code(row, n_outs):
    """비타석 주루 이벤트 행의 KBSA result 코드 (해당 없으면 None).
    WBSC는 도루/도루자/견제사/폭투/포일을 투구 행(pa==0)에 기록한다."""
    nar = str(row.get('narrative', '') or '')
    nl = nar.lower()
    if n_outs:
        if _CS_RE.search(nar) or 'caught stealing' in nl:
            return '도루자'
        if _PO_RE.search(nar) or 'picked off' in nl:
            return '견제사'
        return '주자아웃'
    if 'stolen base' in nl:
        return '도루'
    if row.get('wp') or 'wild pitch' in nl:
        return '폭투'
    if row.get('pb') or 'passed ball' in nl:
        return '포일'
    if row.get('balk') or 'balk' in nl:
        return '보크'
    if _ERR_RE.search(nar) or 'error' in nl:
        return error_code(nar)
    return None


def event_orders(row, n_outs) -> str:
    """후속행의 order(관여 주자 타순들). KBSA는 여러 명이면 타순을 이어붙인다(예 '36').

    아웃 계열(OUT_EVENTS)은 step5_track이 order에 있는 주자를 '전부 루상에서 제거'
    하므로 아웃된 주자만 넣어야 한다. 진루 계열은 움직인 주자를 리드주자부터 넣는다."""
    before, after = bases_before(row), bases_after(row)
    if n_outs:
        outb = [b for b in (3, 2, 1) if row.get(f'runner{b}out') and b in before]
        if not outb:
            # runnerXout 플래그가 없으면 '사라진 주자'로 추정
            outb = [b for b in (3, 2, 1)
                    if b in before and before[b] not in after.values()][:n_outs]
        return ''.join(str(slot_order(before[b])) for b in outb)
    return ''.join(str(slot_order(before[b])) for b in (3, 2, 1)
                   if b in before and before[b] != after.get(b))


# "NAME to 2nd on stolen base." / "NAME scores on E2T." 같은 진루 한 건.
# 원문은 사건마다 이런 문장을 이어 붙인다.
_MOVE_RE = re.compile(
    r"([A-Z][^.]*?)\s+(?:to\s+(1st|2nd|3rd|home)|(scores))\s+on\s+([^.]+?)\.")
_BASE_NUM = {"1st": 1, "2nd": 2, "3rd": 3, "home": 4}
_CAUSE_WORDS = (("stolen base", "도루"), ("wild pitch", "폭투"),
                ("passed ball", "포일"), ("balk", "보크"))


def _cause_code(text: str) -> object:
    """진루 사유 -> KBSA 코드. 모르는 사유면 None."""
    t = text.lower()
    for word, code in _CAUSE_WORDS:
        if word in t:
            return code
    if "error" in t or _ERR_RE.search(text):
        return error_code(text)
    return None


def split_runner_events(row, narrative, slot_map, raw_name):
    """한 행에 겹친 주루 사건을 [(코드, {루: slot} 상태, order, 득점), ...] 로 나눈다.

    원문을 순서대로 읽어 "누가 어디로, 무엇 때문에" 를 모으고, 사유가 바뀌는
    지점에서 끊는다. 사유가 하나뿐이면 빈 목록을 돌려주고 호출부가 예전대로
    한 행을 만든다 — 굳이 같은 결과를 다른 길로 만들 이유가 없다.

    이름 대조는 반드시 치환 전 로마자 이름(raw_name)으로 한다. 원문은 WBSC 가 쓴
    로마자 그대로인데("HOMIN LEE"), 치환된 이름("이호민")과 맞춰 보면 우리 대표팀
    선수만 전부 어긋난다 — 실제로 그래서 한국 경기만 안 나뉘고 있었다."""
    before = bases_before(row)
    if not before:
        return []
    # 이름 -> 지금 있는 루
    where = {}
    for b, slot in before.items():
        nm = raw_name.get(slot_map.get(slot))
        if nm:
            where[str(nm)] = b

    moves = []
    for m in _MOVE_RE.finditer(narrative):
        who, to_base, scores, cause = m.group(1).strip(), m.group(2), m.group(3), m.group(4)
        code = _cause_code(cause)
        if code is None:
            continue
        # 원문의 이름은 앞에 볼카운트 등이 붙어 올 수 있어 뒤에서부터 맞춰 본다.
        hit = next((nm for nm in where if who.endswith(nm)), None)
        if hit is None:
            return []                      # 한 명이라도 못 찾으면 나누지 않는다
        dest = 4 if scores else _BASE_NUM.get(to_base or "", 0)
        if not dest:
            return []
        moves.append((code, hit, dest))

    if len({c for c, _, _ in moves}) < 2:
        return []                          # 사유가 하나면 나눌 것이 없다

    slot_of = {str(raw_name.get(slot_map.get(s))): s for s in before.values()}
    state = dict(before)                   # {루: slot}
    out = []
    for code, mv in _group_moves(moves):
        got = _apply_group(state, mv, slot_of)
        if got is None:
            return []
        orders, runs, scorers = got
        out.append((code, dict(state), orders, runs, scorers))
    return out


# 타석 행에서 떼어낼 사건. 예전에는 타구에 딸린 실책 진루(...on E8T)를 그 타석의
# 결과로 보고 남겼는데, 그러면 "3내안인데 곧장 1·3루" 처럼 안타가 만든 상황과 실책이
# 만든 상황이 한 행에 섞인다(208117 5회 우주로 3내안 뒤 3루수 송구실책으로 남현우 3루).
# 이제 실책 진루도 따로 뗀다. 코드는 _cause_code 가 '{자리}송실' 식으로 준다.
_PA_SPLIT_CAUSES = {"도루", "폭투", "포일", "보크"}

# 이동 한 건. 사유가 없는 이동("NAM to 2nd.")도 읽어야 실책 직전 자리를 안다.
_ANY_MOVE_RE = re.compile(
    r"([A-Z][^.]*?)\s+(?:to\s+(1st|2nd|3rd|home)|(scores))(?:\s+on\s+([^.]+?))?\.")


def _pa_split_code(cause):
    """타석 행에서 떼어낼 사유면 결과 코드, 아니면 None."""
    if not cause or 'obstruction' in cause.lower():   # 주루방해는 실책이 아니다
        return None
    code = _cause_code(cause)
    if code in _PA_SPLIT_CAUSES:
        return code
    if code and (code.endswith('송실') or code.endswith('포실')
                 or code in ('송구실책', '포구실책')):
        return code
    return None


def split_pa_runner_events(row, narrative, slot_map, raw_name, after):
    """타석 행에 섞인 주루 사건(도루·폭투·실책 진루 등)을 떼어낸다.

    돌려주는 값은 (타석 행에 쓸 주자 상태, [(코드, 상태, order, 득점, 득점자), ...]).
    뗄 것이 없거나 확실하지 않으면 None 을 돌려주고 호출부가 예전대로 한 행만
    만든다.

    타석 행의 주자 상태는 최종 상태에서, 떼어낸 사건으로 움직인 선수를 그 사건
    직전 자리로 되돌린 것이다. WBSC 의 runner*after 는 한 플레이에서 일어난 일을 모두
    반영한 값이라, 그대로 쓰면 아직 일어나지 않은 도루·실책 진루까지 타석 행에
    들어간다."""
    before = bases_before(row)
    batter_slot = row.get('lineupslot')

    slot_of, pos = {}, {}
    for b, slot in before.items():
        nm = raw_name.get(slot_map.get(slot))
        if nm:
            slot_of[str(nm)] = slot
            pos[str(nm)] = b
    bname = raw_name.get(slot_map.get(batter_slot, row.get('batterid')))
    if bname and str(bname) not in slot_of:
        slot_of[str(bname)] = batter_slot
        # 타자는 출루한 루에서 출발한다. 안 나갔으면 움직일 일도 없다.
        pos[str(bname)] = (3 if row.get('triple') == 1 else
                           2 if row.get('double') == 1 else 1)

    moves, split_seen = [], set()
    for m in _ANY_MOVE_RE.finditer(narrative):
        who_txt = m.group(1).strip()
        who = next((nm for nm in slot_of if who_txt.endswith(nm)), None)
        code = _pa_split_code(m.group(4))
        if who is None:
            if code:
                return None                  # 떼어낼 사건의 주인을 못 찾으면 손대지 않는다
            continue
        dest = 4 if m.group(3) else _BASE_NUM.get(m.group(2) or "", 0)
        if not dest:
            return None
        if code:
            moves.append((code, who, dest, pos.get(who)))
            split_seen.add(who)
        elif who in split_seen:
            return None                      # 떼어낸 사건 뒤에 또 움직이면 가를 수 없다
        pos[who] = dest
    if not moves:
        return None

    # 떼어낸 사건으로 움직인 선수를 첫 사건 직전 자리로 되돌린다.
    pa_state = dict(after)
    first_src = {}
    for _code, who, _dest, src in moves:
        first_src.setdefault(who, src)
    # 움직인 선수를 모두 뺀 다음에 넣는다. 한 명씩 빼고 넣으면 아직 안 뺀 다른
    # 선수가 그 자리에 있어 겹친 것으로 잘못 본다(207387 2회 HO·CHEUNG).
    moving = {slot_of[w] for w in first_src}
    pa_state = {b: s for b, s in pa_state.items() if s not in moving}
    for who, src in first_src.items():
        if src is None or src == 4:
            return None
        if src in pa_state:
            return None                      # 되돌린 자리가 이미 찼으면 포기한다
        pa_state[src] = slot_of[who]

    state, out = dict(pa_state), []
    for code, mv in _group_moves([(c, w, d) for c, w, d, _ in moves]):
        got = _apply_group(state, mv, slot_of)
        if got is None:
            return None
        orders, runs, scorers = got
        out.append((code, dict(state), orders, runs, scorers))
    if state != dict(after):
        return None                          # 다 적용했는데 최종 상태와 다르면 믿지 않는다
    return pa_state, out




# ── 메인 변환 ────────────────────────────────────────────────────────
def convert_game(game_id: int, orig: dict = None):
    """(KBSA 포맷 PBP DataFrame, 진단 dict) 반환."""
    orig = orig or fetch_game(game_id)
    gd = orig["gameData"]
    ti = orig.get("tournamentInfo", {}) or {}
    name_map, pos_map = build_player_lookup(orig)
    raw_name = dict(name_map)                       # 로마자 원문 (narrative 대조용)
    name_map = apply_korea_name_map(name_map)       # 다음 셀들에서 정의
    slot_map = build_slot_map(orig)
    uni_map = build_uniform_map(orig)

    df = plays_to_df(orig)
    pitcher_tl = build_pitcher_timeline(orig, df)

    away_label, home_label = gd["awaylabel"], gd["homelabel"]
    away_runs, home_runs = gd["awayruns"], gd["homeruns"]
    if away_runs > home_runs:
        result1, result2 = "W", "L"
    elif away_runs < home_runs:
        result1, result2 = "L", "W"
    else:
        result1, result2 = "T", "T"
    date_part, time_part = gd["start"].split(" ")   # "2025-09-14 11:00:00"
    date_str = date_part.replace("-", ".")          # get_month_key() 호환 (YYYY.MM.DD)
    meta = {
        'tournament': ti.get("tournamentname", ""),
        'details': f"{date_str} {time_part[:5]} {gd.get('stadium', '')}",
        'year': int(date_part.split("-")[0]),
        'date': date_str, 'time': time_part[:5], 'stadium': gd.get("stadium", ""),
        'team1': away_label, 'score1': away_runs, 'result1': result1,
        'team2': home_label, 'score2': home_runs, 'result2': result2,
    }

    diag = {'pa_rows': 0, 'event_rows': 0, 'pinch_runner_rows': 0,
            'tiebreak_halves': 0, 'split_runner_outs': 0,
            'stale_batterid': [], 'stale_pitcherid': 0,
            'outs_before_mismatch': [], 'split_events': 0}

    out_rows = []
    cur_bases, cur_half_key = {}, None      # 대주자 행의 on_base_123 채우기용
    for pos_i, (_, row) in enumerate(df.iterrows()):
        half = row["_half"]
        nar = str(row.get("narrative", "") or "").strip()

        base = {c: None for c in KBSA_COLUMNS + RUNNER_COLUMNS}
        base.update(meta)
        base.update({'game_id': game_id,
                     'team': away_label if half == 'top' else home_label,
                     'def_team': home_label if half == 'top' else away_label,
                     'inning': int(row["_inning"]), 'topbot': half,
                     '_playorder': row.get("playorder", 0)})
        pid_pit = pitcher_tl[pos_i]
        # pa==1 행만 센다 — 투구 행의 pitcherid는 원래 자주 비어 있어 의미가 없다
        if row.get("pa") == 1 and pid_pit != row.get("pitcherid"):
            diag['stale_pitcherid'] += 1
        base['pitcher'] = name_map.get(pid_pit)

        n_outs = outs_produced(row)
        runs = int(row.get("runs") or 0)
        score_col = 'away_score' if half == 'top' else 'home_score'

        # 교체/배너 행은 runner 필드가 전부 0이라 주자상황을 못 쓴다 -> 직전 상태 유지
        # (승부치기 배너는 예외 — runner*after에 배치된 주자가 들어있다)
        hk = (int(row["_inning"]), half)
        if hk != cur_half_key:
            cur_bases, cur_half_key = {}, hk
        if is_tiebreak(nar) or not (_SUB_RE.match(nar) or is_banner(nar)):
            cur_bases = bases_after(row)

        # ── 대주자 ──
        m_pr = _PINCH_RUN_RE.match(nar)
        if m_pr:
            info = uni_map.get(('away' if half == 'top' else 'home', m_pr.group(1)))
            if info:
                pid_pr, spot = info
                r = dict(base)
                r.update({'result': '대주자', 'order': spot,
                          'batter': name_map.get(pid_pr),
                          'on_base_123': fmt_bases(cur_bases),
                          'position': pos_map.get(pid_pr),
                          'result_en': nar, 'result_confidence': 'high'})
                r.update(runner_cells(cur_bases, slot_map, name_map))
                out_rows.append(r)
                diag['pinch_runner_rows'] += 1
            continue

        # ── 승부치기(WBSC Tie Break) 주자 배치 ──
        if is_tiebreak(nar):
            aft = bases_after(row)
            # step5_track의 승부주자 처리는 "1루에 놓고 기존 주자를 한 칸씩 밀어올림"
            # 이므로, 최종 상태를 맞추려면 2루 주자를 먼저 내보내야 한다.
            placed = [(b, aft[b]) for b in (2, 1) if b in aft]
            acc = set()
            for _b, slot in placed:
                acc = {x + 1 for x in acc} | {1}
                r = dict(base)
                r.update({'result': '승부주자', 'order': slot_order(slot),
                          'batter': name_map.get(slot_map.get(slot)),
                          'on_base_123': fmt_bases(acc),
                          'result_en': nar, 'result_confidence': 'high'})
                # acc 는 '몇 루가 찼는가' 만 아는 집합이라, 누가 어디 있는지는
                # 배치가 끝난 최종 상태(aft)에서 가져온다.
                r.update(runner_cells(aft, slot_map, name_map))
                out_rows.append(r)
            if placed:
                diag['tiebreak_halves'] += 1
            continue

        if is_banner(nar):
            continue

        # ── 타석 완료 행 ──
        if row.get("pa") == 1:
            # 타자는 batterid가 아니라 lineupslot으로 식별한다(build_slot_map 주석 참고)
            bid = slot_map.get(row.get("lineupslot"), row.get("batterid"))
            if bid != row.get("batterid"):
                diag['stale_batterid'].append(
                    (int(row["_inning"]), half,
                     raw_name.get(row.get("batterid")), raw_name.get(bid)))
            code, conf = classify_result(row)       # 다음 셀에서 정의
            before, after = bases_before(row), bases_after(row)
            # 결과 코드가 이미 주자 아웃을 품고 있으면(병살, 야수선택 R 계열)
            # 따로 '주자아웃' 행을 두지 않는다. KBSA 원본이 그렇게 적는다 —
            # local_mirror 2026 에서 병살 행은 전부 outs_when_up=2, 땅R 행은 전부
            # 1 이고 뒤따르는 주자아웃 행이 없다. 주자아웃 행은 코드만으로는 아웃을
            # 알 수 없을 때만 쓴다(2직 뒤 귀루 실패, 좌안 뒤 주루사 같은 경우).
            # 누가 죽었는지는 out_runner 열이 이미 밝힌다.
            implies_runner_out = '병살' in code or code.endswith('R')
            split = ([] if implies_runner_out
                     else split_runner_outs(row, after, slot_map, raw_name, nar))
            # 타자 본인이 출루한 뒤 루상에서 잡혔으면 그 몫도 주자아웃 행으로 뺀다
            # (208117 4회 엄준상 좌안 뒤 2루 주루사). 기존 주자 아웃으로 설명되지
            # 않는 아웃이 남을 때만 본다 — 아니면 같은 아웃을 두 번 센다.
            if not implies_runner_out:
                runner_outs = sum(1 for _b in (1, 2, 3)
                                  if row.get(f'runner{_b}out') and _b in before)
                if n_outs > runner_outs:
                    taken = set(after) | {_b for _b, _ in split}
                    b_out = batter_out_on_bases(row, nar, bid, raw_name, taken)
                    if b_out:
                        split = list(split) + [b_out]
            batter_nm = name_map.get(bid)

            # 타석이 끝나면서 주자가 따로 움직인 경우(4구 중 도루 등)를 떼어낸다.
            pa_bases, pa_events = after, []
            got = split_pa_runner_events(row, nar, slot_map, raw_name, after)
            if got:
                pa_bases, pa_events = got
                diag['split_events'] += 1
            # 뒤에 '주자아웃' 행으로 뺄 주자는 이 행에서는 아직 루상에 있어야 한다.
            # 타자의 결과와 주자가 죽은 사건은 따로 적기로 했으니, 타석 행에서까지
            # 주자를 치우면 "유직인데 2루 주자가 사라졌다" 가 되고 다음 행이 없는
            # 주자를 잡는다(2026 필리핀전 6회 박보승 유직 + 남현우 주자아웃).
            # after 를 그대로 건드리면 후속행까지 따라 바뀌므로 사본에 채운다.
            if split:
                pa_bases = dict(pa_bases)
                for _b, _slot in split:
                    pa_bases.setdefault(_b, _slot)

            r = dict(base)
            r.update({
                'order': row.get("lineuporder"), 'batter': batter_nm,
                # 플레이가 끝난 뒤 실제로 루상에 남은 주자만 적는다. 예전에는 아웃된
                # 주자를 빈 루에 세워 넣었는데(그래야 KBSA 시뮬레이터가 "이 주자가
                # 죽었다"를 알아챈다는 이유였다), 그 탓에 야수선택·병살 행의 주자
                # 상황이 실제와 달랐다 — 2땅R 에 죽은 주자가 2루에 서 있는 식으로.
                # 누가 죽었는지는 out_runner 열과 '주자아웃' 후속행이 말해 준다.
                'on_base_123': fmt_bases(pa_bases),
                'result': code, 'outs_when_up': (n_outs - len(split)) or None,
                'RBI': int(row.get("rbi") or 0), 'position': pos_map.get(bid),
                'result_en': nar, 'result_confidence': conf,
                'outs_before': int(row.get("outs") or 0),
                # 떼어낸 주루 사건(폭투·도루 등)으로 들어온 주자는 그 행의 득점자다 —
                # 득점 수는 아래서 ev_runs 만큼 빼므로 이름도 같은 기준으로 뺀다.
                'scored_run': _minus_names(
                    scored_names(row, before, slot_map, name_map, batter_nm),
                    {str(name_map.get(slot_map.get(s)))
                     for ev in pa_events for s in ev[4]}),
                'out_runner': out_runner_names(row, before, slot_map, name_map),
            })
            r.update(runner_cells(pa_bases, slot_map, name_map))
            ev_runs = sum(x[3] for x in pa_events)
            if runs - ev_runs:
                r[score_col] = runs - ev_runs
            out_rows.append(r)
            diag['pa_rows'] += 1

            # 후속행의 주자 상황은 이미 아웃이 반영된 실제 상태 그대로다.
            # 아웃된 주자는 out_runner 로 밝힌다 — batter 칸에 넣으면 plate_seq 가
            # 붙어 타순 순환 검사([D])와 타자 성적 집계([F])가 이 행을 타석으로 센다.
            for b, slot in split:
                rr = dict(base)
                nm = name_map.get(slot_map.get(slot))
                # 아웃된 주자는 '있던 루' 로 적는다. split 의 b 는 KBSA 시뮬레이터에
                # 넘기려고 잡아둔 '죽은 루'(narrative 의 out at 2nd)라, 그대로 쓰면
                # 같은 사건을 타석 행은 "1루 황성현", 후속행은 "2루 황성현" 으로
                # 달리 부르게 된다. 원래 어디 있었는지가 사람이 찾는 정보다.
                orig_b = next((k for k, v in before.items() if v == slot), b)
                rr.update({'result': '주자아웃', 'order': slot_order(slot),
                           'on_base_123': fmt_bases(after), 'outs_when_up': 1,
                           'result_en': nar, 'result_confidence': 'high',
                           'outs_before': int(row.get("outs") or 0),
                           'out_runner': f"{orig_b}루 {nm}" if nm else f"{orig_b}루"})
                rr.update(runner_cells(after, slot_map, name_map))
                out_rows.append(rr)
                diag['split_runner_outs'] += 1

            for e_code, e_state, e_order, e_runs, e_scorers in pa_events:
                er = dict(base)
                er.update({'order': e_order, 'on_base_123': fmt_bases(e_state),
                           'result': e_code, 'outs_when_up': None,
                           'result_en': nar, 'result_confidence': 'high',
                           'outs_before': int(row.get("outs") or 0),
                           'scored_run': ", ".join(
                               str(name_map.get(slot_map.get(s))) for s in e_scorers
                           ) or None})
                er.update(runner_cells(e_state, slot_map, name_map))
                if e_runs:
                    er[score_col] = e_runs
                out_rows.append(er)
                diag['event_rows'] += 1
            continue

        # ── 비타석 주루 이벤트 행 ──
        before, after = bases_before(row), bases_after(row)
        if not (n_outs or runs or before != after):
            continue
        code = event_code(row, n_outs)
        if code is None:
            continue

        # 한 투구에 사건이 여러 개 겹쳤으면(도루 뒤 폭투 등) 사건별로 나눈다.
        # 아웃이 난 행은 나누지 않는다 — 아웃 몫이 어느 사건에 붙는지 원문만으로는
        # 가릴 수 없어서, 잘못 나누면 아웃 수가 어긋난다.
        parts = [] if n_outs else split_runner_events(row, nar, slot_map, raw_name)
        if parts:
            for p_code, p_state, p_order, p_runs, p_scorers in parts:
                r = dict(base)
                r.update({'order': p_order, 'on_base_123': fmt_bases(p_state),
                          'result': p_code, 'outs_when_up': None,
                          'result_en': nar, 'result_confidence': 'high',
                          'outs_before': int(row.get("outs") or 0),
                          'scored_run': ", ".join(
                              str(name_map.get(slot_map.get(s))) for s in p_scorers
                          ) or None})
                r.update(runner_cells(p_state, slot_map, name_map))
                if p_runs:
                    r[score_col] = p_runs
                out_rows.append(r)
                diag['event_rows'] += 1
            diag['split_events'] += 1
            continue

        r = dict(base)
        r.update({'order': event_orders(row, n_outs),
                  'on_base_123': fmt_bases(after),
                  'result': code, 'outs_when_up': n_outs or None,
                  'result_en': nar, 'result_confidence': 'high',
                  'outs_before': int(row.get("outs") or 0),
                  'scored_run': scored_names(row, before, slot_map, name_map, None),
                  'out_runner': out_runner_names(row, before, slot_map, name_map)})
        r.update(runner_cells(after, slot_map, name_map))
        if runs:
            r[score_col] = runs
        out_rows.append(r)
        diag['event_rows'] += 1

    pa = pd.DataFrame(out_rows)
    if pa.empty:
        return pa, diag

    # 기존 KBSA 엑셀과 동일하게 원정팀(top) 전체 -> 홈팀(bot) 전체 순으로 재배열
    pa["_half_rank"] = pa["topbot"].map({"top": 0, "bot": 1})
    pa = pa.sort_values(["_half_rank", "inning", "_playorder"], kind="stable").reset_index(drop=True)
    # plate_seq는 "3. 주자상황 체크"의 _is_batting()과 같은 기준으로 부여한다:
    # 실제 타석 행 + 승부주자(KBSA도 타석 순번을 매긴다)에만 주고, 대주자/대타 같은
    # 교체 행과 주루 이벤트 후속행에는 주지 않는다.
    is_bat = (pa["batter"].notna() & pa["batter"].astype(str).str.strip().ne('')
              & ~pa["result"].isin({'대주자', '대타', '대수비'}))
    seq = pd.Series([None] * len(pa), index=pa.index, dtype=object)
    for _team, grp in pa[is_bat].groupby("team", sort=False):
        seq.loc[grp.index] = list(range(1, len(grp) + 1))
    pa["plate_seq"] = seq

    # outs_before 를 최종 행 순서대로 다시 매긴다.
    #
    # WBSC 의 outs 는 '그 플레이 시작 시점' 값이다. 우리는 한 플레이를 여러 행으로
    # 쪼개므로(타석 행 + 주자아웃 후속행), 두 번째 행부터는 그 값이 이미 낡았다 —
    # 앞 행이 만든 아웃이 반영돼 있지 않다. 그대로 두면 "2아웃에서 시작한 타석의
    # 다음 행이 다시 1아웃" 같은 줄이 생겨서, 이 표로 주자 상황을 따라 읽을 수 없다.
    #
    # half 마다 0 에서 시작해 outs_when_up 을 누적한다. half 당 아웃 합이 3 이라는
    # 것은 검수 [E] 가 이닝 요약과 대조해 따로 보증한다.
    orig_outs = pa["outs_before"]
    fixed, acc, key = [], 0, None
    for _, rr in pa.iterrows():
        k = (rr["game_id"], rr["inning"], rr["topbot"])
        if k != key:
            acc, key = 0, k
        fixed.append(acc)
        acc += 0 if pd.isna(rr["outs_when_up"]) else int(rr["outs_when_up"])
    pa["outs_before"] = fixed

    # 타석 행에서는 WBSC 가 준 값과 같아야 한다 — 어긋나면 행을 빠뜨렸거나 순서가
    # 틀렸다는 뜻이라 조용히 넘기면 안 된다.
    is_pa_row = pa["plate_seq"].notna() & pa["result"].ne('승부주자')
    mism = pa[is_pa_row & orig_outs.notna() & (orig_outs != pa["outs_before"])]
    diag['outs_before_mismatch'] = [
        (int(r["inning"]), r["topbot"], r["result"], int(o), int(r["outs_before"]))
        for (_, r), o in zip(mism.iterrows(), orig_outs[mism.index])
    ]

    pa = pa.drop(columns=["_half_rank", "_playorder"])

    if diag['split_events']:
        print(f"  겹친 주루 사건 {diag['split_events']}행을 사건별로 나눔")
    print(f"[체크] 타석 {diag['pa_rows']}행 / 주루이벤트 {diag['event_rows']}행 / "
          f"주자아웃 분리 {diag['split_runner_outs']}행 / 대주자 {diag['pinch_runner_rows']}행 / "
          f"승부치기 half {diag['tiebreak_halves']}개")
    for inn_, hf_, res_, want_, got_ in diag['outs_before_mismatch']:
        print(f"  ⚠ 타석 전 아웃수 불일치: {inn_}회 {hf_} {res_} — "
              f"WBSC {want_} / 누적 {got_}")
    if diag['stale_batterid']:
        for inn, hf, was, now in diag['stale_batterid']:
            print(f"  ⚠ WBSC batterid stale: {inn}회 {hf} {was} -> {now} (lineupslot 기준으로 교정)")
    if diag['stale_pitcherid']:
        print(f"  ⚠ WBSC pitcherid stale {diag['stale_pitcherid']}타석 "
              f"(Pitching Change 추적으로 교정)")
    conf_counts = pa["result_confidence"].value_counts().to_dict()
    print(f"[체크] result 번역 신뢰도: {conf_counts}")

    return pa[KBSA_COLUMNS + EXTRA_REVIEW_COLUMNS + RUNNER_COLUMNS], diag


In [ ]:
"""
WBSC 영문 narrative -> KBSA 한글 결과 코드 번역기

핵심 아이디어: narrative 문장을 자연어로 파싱하는 대신, WBSC가 이미 제공하는
구조화된 필드(strikeout/bb/hbp/roe/sac/sf/gdp/infieldfly/h/double/triple/homerun 등)를
1차 신호로 쓰고, narrative에 박혀 있는 표준 스코어링 코드(F7, L3, 6-3, E5, KS2,
DP 463 등)에서 수비 위치 숫자만 뽑아 KBSA 표기로 변환한다.

표준 스코어링 번호(1=P,2=C,3=1B,4=2B,5=3B,6=SS,7=LF,8=CF,9=RF)가
KBSA 표기(투,포,1,2,3,유,좌,중,우)에 대응한다는 것은 local_mirror 기존 데이터의
땅볼/뜬공/실책 포지션별 빈도 분포(예: 유격수>2루수>3루수>1루수 순, 실제 야구
수비 기회 분포와 일치)로 교차 검증했다.

**생성한 코드는 반드시 KBSA 실데이터에 존재하는 어휘여야 한다.** local_mirror
43개 파일에서 뽑은 203개 코드와 대조해서 다음을 맞췄다:
  - 포수 방해 출루는 '수비방해'가 아니라 '타격방해'다. KBSA에서 '수비방해'는
    OUT_EVENTS(타자/주자 아웃)라, 잘못 쓰면 출루가 아웃으로 뒤집힌다.
  - 야수선택은 주자가 죽었느냐로 코드가 갈린다. 죽었으면 '{pos}땅R'(R_RESULTS,
    "타자 출루 + 주자 아웃"), 아무도 안 죽었으면 '{pos}야선'이다. KBSA 실데이터는
    유땅R 275건 / 유야선 6건으로 죽는 경우가 압도적이라 예전에는 전부 땅R로
    보냈는데, 그러면 표에서 죽은 주자를 찾게 된다(2026 BFA 4경기에서 7건 중 2건).
    '야선'이 step5_track의 R 분기를 못 타던 문제는 R_RESULTS에 야선 계열을
    넣어서 풀었다 — 아웃이 0이면 그 분기가 알아서 _process_hit으로 보낸다.
  - 희생타인데 타자까지 살아나간 경우는 '{pos}희번출'/'{pos}희플출'이다
    (KBSA에 투희번출/포희번출/1희번출/3희번출/좌희플출/중희플출 존재).
    '{pos}희번'(SAC_OUT)으로 쓰면 타자가 루상에 놓이지 않고, 공식기록상 희생타는
    AB가 아니므로 '{pos}실'로 쓰면 AB가 부풀어 오른다(2025 대회에서 8타수 과다).
  - 홈런은 방향에 '월'만 붙인다 — KBSA에 '좌선월홈' 같은 코드는 없다.

한계 (낮은 신뢰도로 표시됨):
  - 2루타/3루타의 정확한 존(선상/갭/펜스 상단) 구분은 완벽하지 않아 'med'로 남긴다
    (예: "doubles to left field" -> 기본값 "좌월2").
  - result_confidence가 high가 아닌 행은 마지막 셀에서 따로 뽑아 검수한다.
"""

import re

# 표준 스코어링 번호(1=P,2=C,3=1B,4=2B,5=3B,6=SS,7=LF,8=CF,9=RF) -> KBSA 표기
STD2KBSA = {1: '투', 2: '포', 3: '1', 4: '2', 5: '3', 6: '유', 7: '좌', 8: '중', 9: '우'}

# step5_track(주자상황 체크)의 R_RESULTS 중 이 변환기가 쓸 수 있는 것들.
# 여기 없는 조합(예 '유번R')을 만들면 R 분기를 못 타고 일반 아웃으로 떨어진다.
R_RESULTS_USABLE = {
    '1땅R', '2땅R', '3땅R', '우땅R', '유땅R', '중땅R', '좌땅R', '투땅R', '포땅R',
    '투번R', '포번R', '1번R', '2번R', '3번R',
}

# 아무도 아웃되지 않은 야수선택에 쓰는 코드. KBSA 어휘에 있는 조합만 넣었다
# ('좌야선'은 KBSA 실데이터에 없어서 뺐다 — 그 경우는 아래에서 땅R로 물러선다).
FC_NO_OUT_USABLE = {
    '1야선', '2야선', '3야선', '우야선', '유야선', '중야선', '투야선', '포야선',
}

# 파울플라이에 쓰는 코드. KBSA 어휘에 있는 조합만 넣었다
# ('중파플'은 중견수 뒤에 파울 지역이 없어 애초에 성립하지 않는다).
# 여기 없는 자리면 아래에서 그냥 뜬공('플')으로 물러선다.
FOUL_FLY_USABLE = {
    '1파플', '2파플', '3파플', '우파플', '유파플', '좌파플', '투파플', '포파플',
}

# 홈런은 방향만 쓰고 '선'(파울라인 쪽) 세분은 KBSA 어휘에 없다
# (KBSA 실데이터: 좌월홈/우월홈/중월홈/좌중홈/우중홈 만 존재).
HR_ZONE_FIX = {'좌선': '좌', '우선': '우'}


def std_pos(digit):
    try:
        return STD2KBSA.get(int(digit))
    except (TypeError, ValueError):
        return None


def leading_num(s):
    if not s:
        return None
    m = re.search(r'(\d)', str(s))
    return m.group(1) if m else None


def _out_at_pos(narrative):
    """'... out at 2nd 16.' 처럼 붙는 수비 연결 숫자에서 첫 자리를 뽑는다."""
    m = re.search(r'out at \w+ (\d)', str(narrative or ''))
    return m.group(1) if m else None


def batter_reached(row) -> bool:
    """이 타석에서 타자 본인이 살아나갔는지 (lineupslot이 타석 후 루상에 있거나
    narrative에 'reaches on'이 있으면 출루)."""
    if row.get('lineupslot') in bases_after(row).values():
        return True
    return 'reaches on' in str(row.get('narrative', '') or '').lower()


DIRECTION_PATTERNS = [
    (r'through the left side', '좌', False),
    (r'through the right side', '우', False),
    (r'up the middle', '중', False),
    (r'to (?:deep )?left[- ]center', '좌중', False),
    (r'to (?:deep )?right[- ]center', '우중', False),
    (r'down (?:the )?left field line', '좌선', False),
    (r'down (?:the )?right field line', '우선', False),
    (r'down lf line', '좌선', False),
    (r'down rf line', '우선', False),
    (r'to (?:deep )?left field', '좌', False),
    (r'to (?:deep )?right field', '우', False),
    (r'to (?:deep )?center field', '중', False),
    (r'to (?:the )?pitcher', '투', True),
    # "singles to home plate area" = 포수 정면 타구 (번트 안타에서 나온다)
    (r'to (?:the )?home plate area', '포', True),
    (r'to (?:the )?catcher', '포', True),
    (r'to (?:the )?first base(?:man)?', '1', True),
    (r'to (?:the )?second base(?:man)?', '2', True),
    (r'to (?:the )?third base(?:man)?', '3', True),
    (r'to (?:the )?shortstop', '유', True),
]


def classify_hit_direction(narrative):
    text = narrative.lower()
    for pat, zone, is_infield in DIRECTION_PATTERNS:
        if re.search(pat, text):
            return zone, is_infield
    return None, False


def hit_code(narrative, bunt, double, triple, homerun):
    zone, is_infield = classify_hit_direction(narrative)
    if zone is None:
        return None, 'low'

    if homerun:
        z = HR_ZONE_FIX.get(zone, zone)
        return (f'{z}월홈' if z in ('좌', '우', '중') else f'{z}홈'), 'med'
    if bunt:
        return f'{zone}번안', 'high'
    if is_infield:
        return f'{zone}내안', 'high'
    if triple:
        if zone.endswith('선') or zone in ('좌중', '우중'):
            return f'{zone}3', 'high'
        return f'{zone}월3', 'med'
    if double:
        if zone.endswith('선') or zone in ('좌중', '우중'):
            return f'{zone}2', 'high'
        return f'{zone}월2', 'med'
    return f'{zone}안', 'high'


def classify_result(row) -> tuple:
    """WBSC gamePlays 한 행(PA 완료 행)을 (KBSA 결과코드, 신뢰도) 로 변환."""
    narrative = str(row.get('narrative', '') or '').strip()
    nl = narrative.lower()
    result3 = row.get('result3')
    result3 = '' if pd.isna(result3) else str(result3).strip()
    bunt = row.get('bunt', 0) == 1

    if row.get('strikeout', 0) == 1:
        # 투스트라이크 뒤 번트 파울은 쓰리번트 규칙으로 삼진이다. WBSC 는 그냥
        # strikeout 으로 주고 원문에만 "2 Strike Bunt Foul (KSB)" 로 남기는데,
        # KBSA 는 이걸 '쓰리번트' 로 따로 적는다.
        if re.search(r'\bKSB\b', narrative) or 'bunt foul' in nl:
            return '쓰리번트', 'high'
        return '삼진', 'high'
    if row.get('ibb', 0) == 1:
        return '고의4구', 'high'
    if row.get('bb', 0) == 1:
        return '4구', 'high'
    if row.get('hbp', 0) == 1:
        return '사구', 'high'
    # WBSC ci = catcher's interference (타자에게 1루 부여, AB 아님).
    # KBSA에서 이 상황의 코드는 '타격방해'(WALK_RESULTS)다 — '수비방해'는 아웃이다.
    if row.get('ci', 0) == 1 or 'catchers interference' in nl:
        return '타격방해', 'high'

    # 인필드플라이/규칙아웃은 타자가 아웃이므로 실책/야수선택 판정보다 먼저 본다.
    if row.get('infieldfly', 0) == 1 or 'infield fly' in nl:
        pos = std_pos(leading_num(result3)) or std_pos(_out_at_pos(narrative))
        return (f'{pos}인플', 'high') if pos else ('인필드플라이', 'low')

    # 희생번트/희생플라이는 실책·야수선택이 겹쳐도 공식기록상 희생타(AB 아님)다.
    # 타자까지 살아나갔으면 '희번출'/'희플출'을 써야 한다(위 docstring 참고).
    is_sac_txt = 'sacrifice' in nl
    reached = row.get('roe', 0) == 1 or 'reaches on' in nl
    if (row.get('sac', 0) == 1 or is_sac_txt) and (bunt or 'bunt' in nl):
        pos = std_pos(leading_num(result3))
        suf = '희번출' if reached else '희번'
        return (f'{pos}{suf}', 'high') if pos else ('희생번트', 'low')
    if row.get('sf', 0) == 1 or (is_sac_txt and not bunt and row.get('h', 0) != 1):
        pos = std_pos(leading_num(result3))
        suf = '희플출' if reached else '희플'
        return (f'{pos}{suf}', 'high') if pos else ('희생플라이', 'low')

    if row.get('roe', 0) == 1:
        pos = std_pos(leading_num(result3))
        return (f'{pos}실', 'high') if pos else ('실책', 'low')

    # 타자 본인은 안타로 살고(h==1) narrative엔 다른 주자의 진루 설명으로
    # "fielders choice"가 섞여 나오는 경우가 있어, h==1 을 먼저 확인해야
    # 번트안타 등을 오분류하지 않는다.
    if row.get('h', 0) == 1:
        code, conf = hit_code(narrative, bunt, row.get('double', 0) == 1,
                              row.get('triple', 0) == 1, row.get('homerun', 0) == 1)
        return (code, conf) if code else (narrative, 'none')

    if 'fielders choice' in nl:
        digits = leading_num(result3) or _out_at_pos(narrative)
        pos = std_pos(digits)
        if not pos:
            return '야수선택', 'low'
        # 야수선택인데 기존 주자가 아무도 안 죽는 경우가 있다 — 야수가 잡으려던
        # 주자가 세이프거나, 송구가 빗나가 다들 살아나가는 식이다. 그때까지 R 코드
        # ("타자 출루 + 주자 아웃")로 적으면 표에서 죽은 주자를 찾게 된다.
        if not any(row.get(f'runner{b}out') for b in (1, 2, 3)):
            fc = f'{pos}야선'
            return (fc, 'high') if fc in FC_NO_OUT_USABLE else (f'{pos}땅R', 'med')
        bunt_r = f'{pos}번R'
        return ((bunt_r if (bunt or 'bunt' in nl) and bunt_r in R_RESULTS_USABLE
                 else f'{pos}땅R'), 'high')

    if row.get('gdp', 0) == 1:
        # gdp 플래그가 있어도 타자 본인은 실책으로 살아나가는 경우가 있다
        # (예: "reaches on GDP muffed catch error"). '병살타'로 찍으면 타자가
        # 아웃된 것처럼 처리돼 주자 정체가 어긋나므로 실책 출루로 본다.
        chain = result3.replace('DP', '').strip()
        pos = std_pos(leading_num(chain)) or std_pos(_out_at_pos(narrative))
        if batter_reached(row):
            return (f'{pos}실', 'high') if pos else ('실책', 'low')
        # 번트 병살. KBSA 는 '투땅병살[BUNT]' 처럼 태그로 적지만 국가대표는
        # '포번병살' 처럼 번트임을 코드에 넣는다(국가대표 전용 — 검수 허용목록).
        if bunt or 'bunt' in nl:
            return (f'{pos}번병살', 'high') if pos else ('병살타', 'low')
        return (f'{pos}땅병살', 'high') if pos else ('병살타', 'low')

    # result3 필드는 라인드라이브/뜬공을 구분하지 않고 둘 다 'F<n>'로 저장한다.
    # narrative 원문에만 "L<n>" 표기가 남아있으므로 여기서 직접 파싱한다.
    m_line = re.search(r'\bL(\d)\b', narrative)
    if m_line and row.get('flyout', 0) == 1:
        pos = std_pos(m_line.group(1))
        return (f'{pos}직', 'high') if pos else (narrative, 'low')

    # 파울플라이도 마찬가지다. result3 는 파울 여부를 담지 않아 평범한 뜬공과
    # 구분이 안 되고, 원문에만 FF<n>(파울플라이) / FP<n>(파울 팝플라이)로 남는다
    # (예: "JURO WOO flies out foul. FF3."). 원문의 "foul"이라는 낱말만 보면
    # 투구 기록의 "Foul." 까지 걸리므로 표준 스코어 기호를 본다.
    m_foul = re.search(r'\bF[FP](\d)\b', narrative)
    if m_foul and row.get('flyout', 0) == 1:
        pos = std_pos(m_foul.group(1))
        if f'{pos}파플' in FOUL_FLY_USABLE:
            return f'{pos}파플', 'high'

    if result3:
        letter = result3[0] if result3[0].isalpha() else ''
        if letter in ('F', 'L'):
            pos = std_pos(leading_num(result3))
            suffix = '플' if letter == 'F' else '직'
            return (f'{pos}{suffix}', 'high') if pos else (narrative, 'low')
        pos = std_pos(leading_num(result3))
        if row.get('groundout', 0) == 1:
            return (f'{pos}땅', 'high') if pos else ('땅볼아웃', 'low')
        if pos:
            return f'{pos}땅R', 'med'

    return narrative, 'none'


In [ ]:
"""
한국 선수 로마자 이름 -> 한글 이름 매핑

WBSC 데이터에는 로마자 표기(firstname/lastname)만 있고 한글 원본이 없어서
자동 역변환이 불가능하다 (로마자->한글은 다대일 문제: 예를 들어 "Jaewon"이
"재원"인지 "재훤"인지 로마자만으로는 확정할 수 없음).

아래 20명은 2025 U-18 월드컵 경기들의 boxScore를 스캔해 얻은 한국 대표팀
로스터 전체(등번호 포함)다.

뉴스 기사로 교차 확인된 항목:
  23 OH Jaewon      -> 오재원 (유신고, 주장)
  28 KIM Geonhwi    -> 김건휘 (충암고)
  30 PARK Jiho      -> 박지호 (군산상일고)
  17 AN Jiwon       -> 안지원 (부산고)
   5 EOM Junsang    -> 엄준상 (덕수고)
  29 HA Hyunseung   -> 하현승 (부산고)

⚠ 아래 14명은 로마자 표기에서 **추정**해 채워 넣은 값이고 아직 1차 출처로
확인되지 않았다. 그대로 batter_box_global / pitcher_box_global 과 웹까지
흘러가므로, 대한야구소프트볼협회(KBSA) 2025-07-11 로스터 발표나 나무위키
"대한민국 U-18 야구 국가대표팀/2025"로 반드시 재확인해야 한다:
   3 SHIN Jaein     -> 신재인
  21 KIM Joel       -> 김요엘   (기사에서 "김요엘"로 언급됨 — 거의 확실)
   1 LEE Taeyang    -> 이태양
   6 PARK Hangyeol  -> 박한결
   7 LEE Yeonwoo    -> 이연우
  10 LEE Huiseong   -> 이희성
  11 CHOI Yohan     -> 최요한
  12 HEO Yun        -> 허윤
  13 PARK Jisung    -> 박지성
  15 PARK Junsung   -> 박준성
  16 KIM Jiseok     -> 김지석
  18 PARK Junhyun   -> 박준현
  19 SHIN Dongkun   -> 신동균  (로마자로는 "신동근"이 더 자연스러움 — 특히 의심)
  27 KANG Minkee    -> 강민기

출처: 네이트 스포츠 "'김요엘 10K' ... 슈퍼라운드 진출"(20250909),
"캡틴 오재원, 1선발 박석민 주니어, 투타겸업 신재인까지..."(20250827),
2025 로스터 발표 기사, 그 외 뉴스 검색 스니펫(안지원 부산고 6번타자 우익수).
페이스북 게시물은 이미지라 OCR 불가, namu.wiki는 자동 접근(WebFetch)이 403.

값을 None으로 두면 apply_korea_name_map()이 로마자 이름을 그대로 남긴다.

2026 BFA 분은 이런 추정이 필요 없다 — KBSA 발표 명단이 이 저장소
web/data/national_team.json 에 한글 이름째로 들어 있어서 그대로 쓴다.
앞으로도 대표팀 대회는 그 파일을 먼저 보는 게 맞다.
"""

KOREA_NAME_MAP_2025 = {
    "OH Jaewon": "오재원",
    "KIM Geonhwi": "김건휘",
    "PARK Jiho": "박지호",
    "AN Jiwon": "안지원",
    "EOM Junsang": "엄준상",
    "HA Hyunseung": "하현승",
    "SHIN Jaein": "신재인",       # 재확인 권장
    "KIM Joel": "김요엘",         # 재확인 권장

    # 아래는 미확인 — 한글 이름을 확인하는 대로 값만 채우면 됨
    "LEE Taeyang": "이태양",
    "PARK Hangyeol": "박한결",
    "LEE Yeonwoo": "이연우",
    "LEE Huiseong": "이희성",
    "CHOI Yohan": "최요한",
    "HEO Yun": "허윤",
    "PARK Jisung": "박지성",
    "PARK Junsung": "박준성",
    "KIM Jiseok": "김지석",
    "PARK Junhyun": "박준현",
    "SHIN Dongkun": "신동균",
    "KANG Minkee": "강민기",
}

# 2026 BFA U-18 (18명). 2025와 달리 로마자에서 추정한 값이 아니라,
# 저장소 안의 1차 출처 web/data/national_team.json(KBSA 대표팀 발표 명단)에
# 있는 한글 이름을 그대로 가져왔다. WBSC 팀 페이지의 18명과 KBSA 19명을
# 성씨+포지션으로 맞춰 18명 전원이 후보 1명으로 떨어졌고(모호 0건),
# 투타도 17명이 일치한다. 다만 WBSC 의 B/T 열은 BAT/THROW 순이라 우리
# 투타(투/타)와 앞뒤가 반대다 — KBSA 우투좌타 = 타 L / 투 R = "L/R".
# 그 기준으로 읽으면 18명 중 17명이 맞고(같은 우투좌타인 강인규·안우석·
# 전영훈이 모두 L/R), 윤예성만 R/L 로 올라와 있다. 우리 순서로 읽으면
# 오히려 15명만 맞으니 WBSC 쪽 한 건이 틀린 것이다 — 인창고 윤예성은
# 선수 프로필(personNo 201902002665)에도 우투좌타다. 이름 매칭에는
# 영향이 없다.
# KBSA 명단 19명 중 김지우(서울고 내야수)만 WBSC 18인 엔트리에 없다.
# 나중에 추가 엔트리로 나와도 이름이 로마자로 남지 않도록 같이 넣어 둔다.
KOREA_NAME_MAP_2026 = {
    "PARK Boseung": "박보승",         # 경남고 3학년 외야수
    "GWAK Dohyeon": "곽도현",         # 부산공업고 3학년 투수
    "PARK Geunseo": "박근서",         # 서울디자인고 3학년 투수
    "LEE Homin": "이호민",            # 경남고 3학년 내야수
    "CHO Huiseong": "조희성",         # 유신고 3학년 외야수
    "NAM Hyeonu": "남현우",           # 서울컨벤션고 3학년 내야수
    "HA Hyunseung": "하현승",         # 부산고 3학년 투수
    "KANG Inkyu": "강인규",           # 청주고 3학년 내야수
    "WON Jiwoo": "원지우",            # 강릉고 3학년 포수
    "EOM Junsang": "엄준상",          # 덕수고 3학년 내야수
    "WOO Juro": "우주로",             # 대전고 3학년 내야수
    "HAN Kyumin": "한규민",           # 대전고 2학년 투수
    "KIM Minhun": "김민훈",           # 광주진흥고 3학년 투수
    "JANG Minje": "장민제",           # 충암고 3학년 외야수
    "HWANG Seonghyun": "황성현",      # 덕수고 3학년 외야수
    "AN Wooseok": "안우석",           # 경남고 3학년 내야수
    "YOON Yesung": "윤예성",          # 인창고 3학년 투수
    "JEON Younghoon": "전영훈",       # 세광고 2학년 포수
    "KIM Jiwoo": "김지우",              # 서울고 3학년 내야수 (WBSC 엔트리 외)
}

# 두 대회를 한 맵으로 합쳐 쓴다. 엄준상·하현승은 두 해 모두 뽑혀
# 키가 겹치는데 같은 사람이라 값도 같다.
KOREA_NAME_MAP = {**KOREA_NAME_MAP_2025, **KOREA_NAME_MAP_2026}


def _name_key(name) -> tuple:
    """대소문자와 성·이름 순서를 무시한 열쇠.

    대회마다 표기가 다르다. 2025 월드컵(wbsc.org)은 다른 나라와 같이 성을 앞에
    두고 대문자로 적었는데("HA Hyunseung"), 2026 BFA(wbscasia.org)는 한국만
    이름이 앞에 오고 둘 다 대문자다("HYUNSEUNG HA") — 같은 대회의 일본은
    "FUKUSHIMA Hinata" 로 멀쩡하니 우리 협회가 넣은 자료가 그런 것 같다.

    대회마다 표기를 새로 등록하면 다음 대회에서 또 뚫린다. 토큰을 정렬해 한
    열쇠로 모으면 순서가 어떻든 맞는다. 대표팀 명단 안에서 이 열쇠가 겹치지
    않는 것은 아래에서 확인한다 — 겹치면 그 자리에서 죽는 편이 낫다.
    조용히 로마자로 흘러가면 검수 [H] 가 잡을 때까지 웹까지 간다."""
    return tuple(sorted(str(name).upper().split()))


KOREA_NAME_LOOKUP: dict = {}
for _k, _v in KOREA_NAME_MAP.items():
    _key = _name_key(_k)
    if KOREA_NAME_LOOKUP.setdefault(_key, _v) != _v:
        raise ValueError(f"이름 열쇠 충돌: {_k} -> {_v} / 이미 {KOREA_NAME_LOOKUP[_key]}")


def apply_korea_name_map(name_map: dict) -> dict:
    """name_map(playerid -> 로마자 이름)에서 우리 대표팀 선수만 한글로 바꾼다.

    적어둔 표기와 정확히 같으면 그대로 쓰고, 아니면 순서·대소문자를 무시한
    열쇠로 한 번 더 찾는다. 둘 다 없으면 로마자 그대로 둔다(검수 [H] 가 잡는다)."""
    out = dict(name_map)
    for pid, romanized in name_map.items():
        hangul = KOREA_NAME_MAP.get(romanized) or KOREA_NAME_LOOKUP.get(_name_key(romanized))
        if hangul:
            out[pid] = hangul
    return out

In [ ]:
"""
inning_summary_global_{YEAR}.xlsx / batter_box_global_{YEAR}.xlsx / pitcher_box_global_{YEAR}.xlsx

KBSA 파이프라인은 크롤링 단계(1번 노트북)에서 inning_summary_{YEAR}.xlsx /
batter_box_{YEAR}.xlsx / pitcher_box_{YEAR}.xlsx 를 "실제값" 원본으로 별도
크롤링해두고, 나중에 검수 단계(2/3번 노트북)에서 PBP로부터 계산한 값과 그
실제값을 대조(diff)한다. 여기서도 동일한 역할 분담을 따른다:

  - 이 노트북(크롤링)의 역할: "실제값" 파일 3개를 만든다.
  - "WBSC 검수.ipynb"의 역할: 이 파일들과 PBP를 대조해서 diff 파일을 만든다.

WBSC는 이 "실제값"을 따로 스크래핑할 필요가 없다 — gameData/boxScore 안에
이미 이닝별 점수(runshome{n}/runsaway{n})와 선수별 공식 기록(AB/H/R/RBI,
투수 BF/ER 등)이 authoritative 값으로 들어있기 때문이다.
"""

def _team_label(orig, teamid):
    gd = orig["gameData"]
    if teamid == gd.get("homeid"):
        return gd["homelabel"]
    if teamid == gd.get("awayid"):
        return gd["awaylabel"]
    return None


def _opponent_label(orig, teamid):
    gd = orig["gameData"]
    if teamid == gd.get("homeid"):
        return gd["awaylabel"]
    if teamid == gd.get("awayid"):
        return gd["homelabel"]
    return None


def build_inning_summary_global(orig: dict, full_df: pd.DataFrame = None) -> pd.DataFrame:
    """이닝별 실제 득점/아웃/팀합계(R/H/E/B)."""
    gd = orig["gameData"]
    game_id = gd["id"]
    away_label, home_label = gd["awaylabel"], gd["homelabel"]
    max_inning = gd.get("innings") or 0

    if full_df is None:
        full_df = plays_to_df(orig)

    # 배너 행(GAME OVER / Middle of the Nth 등)을 제외한 '실제 플레이 행'만 본다.
    # 콜드게임으로 아예 치르지 않은 말공격에도 "GAME OVER<br>AUS wins 15-3" 배너
    # 행이 1개 들어있어서, 단순히 half가 비었는지만 보면 유령 half가 생긴다
    # (2025 대회 173042/173049/173067의 5회말이 score=0/real_out=0으로 만들어졌다).
    # 마지막 실측 지점(끝내기로 3아웃 미만인 half)도 배너 행이 아니라 실제 마지막
    # 플레이 기준으로 잡아야 한다.
    real = full_df[~full_df["narrative"].apply(is_banner)]
    if real.empty:
        return pd.DataFrame()
    last_row = real.iloc[-1]
    last_inning, last_half = int(last_row["_inning"]), last_row["_half"]

    rows = []
    for half, team, run_prefix, R, H, E, B in [
        ("top", away_label, "runsaway", gd["awayruns"], gd["awayhits"], gd["awayerrors"], gd["awaylob"]),
        ("bot", home_label, "runshome", gd["homeruns"], gd["homehits"], gd["homeerrors"], gd["homelob"]),
    ]:
        for inning in range(1, max_inning + 1):
            half_all = real[(real["_inning"] == inning) & (real["_half"] == half)]
            if half_all.empty:
                continue
            score = gd.get(f"{run_prefix}{inning}", 0) or 0
            if inning == last_inning and half == last_half:
                real_out = int(half_all.apply(outs_produced, axis=1).sum())
            else:
                real_out = 3
            rows.append({
                "game_id": game_id, "team1": away_label, "team2": home_label,
                "team": team, "topbot": half, "inning": inning,
                "score": score, "real_out": real_out,
                "R": R, "H": H, "E": E, "B": B,
            })
    return pd.DataFrame(rows)


def build_batter_box_global(orig: dict) -> pd.DataFrame:
    """선수별 공식 타격 기록 (boxScore 그대로, 별도 계산/시뮬레이션 없음)."""
    gd = orig["gameData"]
    game_id = gd["id"]
    name_map = apply_korea_name_map(build_player_lookup(orig)[0])
    rows = []
    for _side, spot, _i, p in _iter_box_players(orig):
        pid = p["playerid"]
        rows.append({
            "game_id": game_id,
            "team": _team_label(orig, p.get("teamid")),
            "def_team": _opponent_label(orig, p.get("teamid")),
            "uniform": p.get("uniform"),
            "batter": name_map.get(pid, f"{p.get('lastname', '')} {p.get('firstname', '')}".strip()),
            "position": p.get("pos"),
            "spot": spot,
            "PA": p.get("pa", 0), "AB": p.get("ab", 0), "H": p.get("h", 0),
            "R": p.get("r", 0), "RBI": p.get("rbi", 0),
            "2B": p.get("double", 0), "3B": p.get("triple", 0), "HR": p.get("hr", 0),
            "BB": p.get("bb", 0), "IBB": p.get("ibb", 0), "SO": p.get("so", 0),
            "HBP": p.get("hbp", 0), "SF": p.get("sf", 0), "SH": p.get("sh", 0),
            "SB": p.get("sb", 0), "CS": p.get("cs", 0), "E": p.get("field_e", 0),
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.drop_duplicates(subset=["game_id", "uniform", "team"]).reset_index(drop=True)


def count_pitches(orig: dict) -> dict:
    """{pitcherid: 투구수}.

    boxScore 에는 투구수가 없다. gamePlays 가 투구 하나를 한 행으로 남기고
    nopitch 로 투구가 아닌 행을 표시하므로(견제·코칭 방문·이닝 배너), 그걸 빼고
    센다. 2026 한국-대만에서 견제 16행이 전부 nopitch=2 로 걸러졌다."""
    out: dict = {}
    for _inn, halves in (orig.get("gamePlays", {}).get("all") or {}).items():
        for _half, plays in halves.items():
            for p in plays:
                pid = p.get("pitcherid")
                if pid and not p.get("nopitch"):
                    out[pid] = out.get(pid, 0) + 1
    return out


def build_pitcher_box_global(orig: dict) -> pd.DataFrame:
    """실제로 등판한(pitch_bf>0) 선수의 공식 투구 기록 (boxScore 그대로).

    NP(투구수)만 boxScore 가 아니라 gamePlays 를 세어 붙인다."""
    gd = orig["gameData"]
    game_id = gd["id"]
    name_map = apply_korea_name_map(build_player_lookup(orig)[0])
    np_map = count_pitches(orig)
    rows = []
    for _side, _spot, _i, p in _iter_box_players(orig):
        bf = p.get("pitch_bf", 0) or 0
        if bf <= 0:
            continue
        pid = p["playerid"]
        rows.append({
            "game_id": game_id,
            "team": _team_label(orig, p.get("teamid")),
            "def_team": _opponent_label(orig, p.get("teamid")),
            "uniform": p.get("uniform"),
            "pitcher": name_map.get(pid, f"{p.get('lastname', '')} {p.get('firstname', '')}".strip()),
            "IP": p.get("pitch_ip"), "BF": bf, "AB": p.get("pitch_ab", 0),
            "H": p.get("pitch_h", 0), "R": p.get("pitch_r", 0), "ER": p.get("pitch_er", 0),
            "BB": p.get("pitch_bb", 0), "IBB": p.get("pitch_ibb", 0),
            "HBP": p.get("pitch_hbp", 0), "SO": p.get("pitch_so", 0), "HR": p.get("pitch_hr", 0),
            "WP": p.get("pitch_wp", 0), "BK": p.get("pitch_bk", 0),
            "W": p.get("pitch_win", 0), "L": p.get("pitch_loss", 0),
            "SV": p.get("pitch_save", 0), "GS": p.get("pitch_gs", 0),
            "NP": np_map.get(pid, 0),
        })
    return pd.DataFrame(rows)


def _save_global(new_rows: pd.DataFrame, path, sort_cols) -> pd.DataFrame:
    """game_id 기준 누적 저장 (같은 game_id 재실행하면 덮어씀, 중복 안 쌓임)."""
    if path.exists():
        existing = pd.read_excel(path)
        existing = existing[~existing["game_id"].isin(new_rows["game_id"].unique())]
        combined = pd.concat([existing, new_rows], ignore_index=True)
    else:
        combined = new_rows.copy()
    combined = combined.sort_values(sort_cols, kind="stable").reset_index(drop=True)
    combined.to_excel(path, index=False)
    return combined


def save_inning_summary_global(new_rows: pd.DataFrame, year: int) -> pd.DataFrame:
    combined = new_rows.copy()
    half_rank = {"top": 0, "bot": 1}
    combined["_half_rank"] = combined["topbot"].map(half_rank)
    combined = _save_global(combined, OUT / f"inning_summary_global_{year}.xlsx",
                             ["game_id", "_half_rank", "inning"])
    return combined.drop(columns=["_half_rank"])


def save_batter_box_global(new_rows: pd.DataFrame, year: int) -> pd.DataFrame:
    return _save_global(new_rows, OUT / f"batter_box_global_{year}.xlsx", ["game_id", "team", "spot"])


def save_pitcher_box_global(new_rows: pd.DataFrame, year: int) -> pd.DataFrame:
    return _save_global(new_rows, OUT / f"pitcher_box_global_{year}.xlsx", ["game_id", "team"])


In [ ]:
"""
PBP 파일 저장: {month_key}_global_{letter}.xlsx (예: 2025_09_global_a.xlsx)

KBSA 1번 노트북의 save_games_to_files()/get_file_path()/next_letter()와 동일한
방식이다: 경기 날짜(date 열, "YYYY.MM.DD")에서 월을 뽑아 그 달의 마지막 파일에
이어붙이고, 10경기(GAMES_PER_FILE)가 차면 다음 알파벳 파일로 넘어간다.
game_id 하나당 파일 하나였던 예전 방식과 달리, 여러 경기가 한 파일에 누적된다.
"""

ALPHABET = "abcdefghijklmnopqrstuvwxyz"
GAMES_PER_FILE = 10


def get_month_key(date_str: str) -> str:
    parts = date_str.split(".")
    return f"{parts[0]}_{parts[1]}"


def get_file_path(month_key: str, letter: str) -> pathlib.Path:
    return OUT / f"{month_key}_global_{letter}.xlsx"


def next_letter(letter: str) -> str:
    idx = ALPHABET.index(letter)
    return ALPHABET[idx + 1]


def get_last_file_in_month(month_key: str):
    """이미 존재하는 파일 중 알파벳상 가장 마지막 파일 경로 (없으면 None)."""
    existing = sorted(OUT.glob(f"{month_key}_global_[a-z].xlsx"))
    return existing[-1] if existing else None


def load_existing_file(path) -> pd.DataFrame:
    if path is not None and path.exists():
        return pd.read_excel(path)
    return pd.DataFrame()


def save_pbp_games(game_data_list) -> None:
    """game_data_list: [(game_id, pbp_df), ...].

    월별로 그룹핑해서 10경기(GAMES_PER_FILE)씩 알파벳 파일에 나눠 담는다.

    재실행(같은 game_id 재크롤링) 처리: 먼저 **그 달의 모든 기존 파일에서** 해당
    game_id 행을 제거하고, 그 다음 빈자리가 있는 파일부터 채운다.
    (예전 버전은 그 달의 마지막 파일만 보고 시작해서, 이미 10경기로 꽉 찬 파일이
    있으면 곧바로 다음 알파벳으로 넘어갔다 — 50경기를 전부 재크롤링하면 a~e에
    낡은 데이터가 그대로 남고 f~j가 새로 생겨 데이터가 두 벌이 됐다.)"""
    month_groups = {}
    for game_id, df in game_data_list:
        month_key = get_month_key(df["date"].iloc[0])
        month_groups.setdefault(month_key, []).append((game_id, df))

    for month_key in sorted(month_groups.keys()):
        games = month_groups[month_key]
        incoming = {gid for gid, _ in games}

        # 1) 그 달 모든 파일에서 재크롤링 대상 game_id 제거
        buckets = {}          # letter -> DataFrame
        for path in sorted(OUT.glob(f"{month_key}_global_[a-z].xlsx")):
            letter = path.stem.split("_")[-1]
            d = load_existing_file(path)
            if not d.empty:
                d = d[~d["game_id"].isin(incoming)]
            buckets[letter] = d

        # 2) 빈자리 있는 파일부터 채우기
        letter = "a"
        for game_id, game_df in games:
            while True:
                cur = buckets.get(letter, pd.DataFrame())
                cnt = cur["game_id"].nunique() if not cur.empty else 0
                if cnt < GAMES_PER_FILE:
                    break
                letter = next_letter(letter)
            buckets[letter] = pd.concat([cur, game_df], ignore_index=True)

        # 3) 저장 (내용이 사라진 파일은 지운다)
        for letter, d in sorted(buckets.items()):
            path = get_file_path(month_key, letter)
            if d.empty:
                if path.exists():
                    path.unlink()
                    print(f"  삭제(내용 없음): {path.name}")
                continue
            d.to_excel(path, index=False)
            n = d["game_id"].nunique()
            flag = "" if n >= GAMES_PER_FILE else ", 미완료"
            print(f"  저장 완료: {path.name} ({n}경기{flag})")


In [ ]:
"""
game_id 자동 수집: GAME_IDS를 비워두면 EVENT(대회) 전체의 game_id를 자동으로 가져온다.

WBSC 일정 페이지(/schedule-and-results)에도 box-score 페이지와 동일하게
data-page JSON이 박혀 있고, props.games 에 그 대회의 경기 전체(대진표) 목록이
들어있다 — 각 경기의 id(=game_idx)와 gamestatustext("F"=종료)를 바로 읽을 수
있어서 KBSA처럼 달력을 월별로 순회할 필요가 없다.

종료('F')가 아닌 경기는 gamePlays.all 자체가 없어서 스크래핑이 불가능하다.
다만 조용히 버리면 안 된다 — 2025 대회 173045(한국-남아공, 09-07)는 status가
'B3'(3회말)인 채로 남아 있어서 전체 51경기 중 1경기가 통째로 누락됐고, 아무
경고도 나오지 않았다. 그래서 아래에서 스킵 목록을 명시적으로 출력한다.
"""

def fetch_schedule() -> list:
    """EVENT 의 경기 전체(대진표). 예정 경기까지 다 들어 있다."""
    url = f"{HOST}/en/events/{EVENT}/schedule-and-results"
    r = creq.get(url, impersonate="chrome", timeout=30)
    r.raise_for_status()
    m = re.search(r'data-page="([^"]+)"', r.text)
    if not m:
        raise RuntimeError("일정 페이지에서 data-page를 찾을 수 없음")
    return json.loads(html.unescape(m.group(1)))["props"]["games"]


# 일정에서 웹까지 들고 갈 값만 추린다. props.games 한 건에는 200개 넘는 열이
# 붙어 있어서 통째로 저장하면 무엇을 쓰는지 알아보기 어렵다.
SCHEDULE_FIELDS = ["id", "gamestatustext", "start", "stadium", "gametypelabel",
                   "awaylabel", "homelabel", "awayruns", "homeruns", "innings"]


def save_schedule(year: int) -> list:
    """일정을 wbsc/schedule_{year}.json 으로 떨군다.

    예정 경기는 gamePlays 가 없어 스크래핑할 게 없지만, 다음 상대가 누구인지는
    여기 있다. export 가 로컬 파일만 읽도록 네트워크는 크롤러가 전담한다."""
    games = fetch_schedule()
    rows = [{k: g.get(k) for k in SCHEDULE_FIELDS} for g in games]
    rows.sort(key=lambda r: (str(r.get("start") or ""), r.get("id") or 0))
    path = OUT / f"schedule_{year}.json"
    path.write_text(json.dumps(rows, ensure_ascii=False, indent=1), encoding="utf-8")
    done = sum(1 for r in rows if r.get("gamestatustext") == "F")
    print(f"  일정 저장: {path.name} ({len(rows)}경기 / 종료 {done})")
    return rows


def fetch_standings() -> list:
    """대회 순위표. [{"group": 조 이름, "rows": [{순위·팀·승·패·무·승률·게임차}]}, ...]

    이 페이지만 data-page JSON 이 없어 HTML 을 읽는다. 표 바로 앞 텍스트에 조
    이름이 들어 있어서 그걸 제목으로 쓴다 — 예선은 Group A/B, 이후는
    Super Round Group / Placement Round Group 이다."""
    url = f"{HOST}/en/events/{EVENT}/standings"
    r = creq.get(url, impersonate="chrome", timeout=30)
    r.raise_for_status()
    parts = re.split(r"<table", r.text)

    def cells(tr):
        return [" ".join(html.unescape(re.sub(r"<[^>]+>", " ", c)).split())
                for c in re.findall(r"<t[dh][^>]*>(.*?)</t[dh]>", tr, re.S)]

    out = []
    for i, part in enumerate(parts[1:], 1):
        head = re.findall(r">([^<>]{3,60})<", parts[i - 1][-700:])
        head = [h.strip() for h in head if h.strip() and not h.strip().startswith("&")]
        group = head[-1] if head else f"Group {i}"

        rows = []
        for tr in re.findall(r"<tr[^>]*>(.*?)</tr>", part, re.S):
            c = [x for x in cells(tr) if x]
            # 팀 칸은 "KOR Korea" 처럼 약자와 이름이 붙어 온다.
            if len(c) < 6 or not c[0].isdigit():
                continue
            # 대회가 끝나면 맨 위에 최종 순위표가 붙는데, 팀 칸 뒤에 "KOR Roster
            # Stats: …" 링크 칸이 끼고 승률·게임차가 없어 조별 표로 읽으면 승수
            # 자리에 글자가 들어온다. 승·패·무가 숫자인 행만 조별 순위로 받는다.
            if not all(x.isdigit() for x in c[2:5]):
                continue
            team = c[1]
            m = re.match(r"^[A-Z]{3}\s+(.*)$", team)
            rows.append({
                "rank": int(c[0]),
                "team": m.group(1) if m else team,
                "W": int(c[2]), "L": int(c[3]), "T": int(c[4]),
                "PCT": c[5], "GB": c[6] if len(c) > 6 else None,
            })
        if rows:
            out.append({"group": group, "rows": rows})
    return out


def save_standings(year: int) -> list:
    """순위표를 wbsc/standings_{year}.json 으로 떨군다."""
    tables = fetch_standings()
    path = OUT / f"standings_{year}.json"
    path.write_text(json.dumps(tables, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"  순위표 저장: {path.name} ({len(tables)}개 조)")
    return tables


def discover_game_ids(statuses=("F",), verbose=True) -> list:
    """EVENT에 속한 전체 game_id 중 statuses에 해당하는 것만 반환.
    기본값 'F'(종료)만 가져오고, 스킵한 경기는 목록으로 알려준다."""
    games = fetch_schedule()

    picked = sorted(g["id"] for g in games if g.get("gamestatustext") in statuses)
    skipped = [g for g in games if g.get("gamestatustext") not in statuses]
    if verbose and skipped:
        print(f"⚠ 미종료/비정상 상태로 스킵한 경기 {len(skipped)}건 "
              f"(전체 {len(games)}경기 중 {len(picked)}경기만 수집):")
        for g in skipped:
            print(f"   game_id={g['id']} status={g.get('gamestatustext')!r} "
                  f"{g.get('awaylabel')} vs {g.get('homelabel')} {g.get('start')}")
    return picked


In [ ]:
YEAR = 2026      # inning_summary_global_{YEAR}.xlsx 등 파일명에 쓸 연도
GAME_IDS = []    # 비워두면 discover_game_ids()로 EVENT의 종료 경기를 전부 자동 수집.
                 # 특정 경기만 하고 싶으면 예: [207380, 207377] 처럼 직접 나열.

if not GAME_IDS:
    GAME_IDS = discover_game_ids()
    print(f"GAME_IDS 비어있음 -> 자동 수집: {len(GAME_IDS)}경기 ({GAME_IDS[0]}~{GAME_IDS[-1]})")

all_pbp = []
all_inning_rows = []
all_batter_box = []
all_pitcher_box = []
all_diag = {}
failed_ids = []

for gid in GAME_IDS:
    print(f"\n=== game_id={gid} ===")
    try:
        orig = fetch_game(gid)                    # 한 번만 요청해서 아래 전부가 공유
        if "all" not in (orig.get("gamePlays") or {}):
            raise RuntimeError("gamePlays.all 없음 (중단/미개최 경기)")
        full_df = plays_to_df(orig)

        pbp_df, diag = convert_game(gid, orig=orig)
        inning_df = build_inning_summary_global(orig, full_df=full_df)
        batter_df = build_batter_box_global(orig)
        pitcher_df = build_pitcher_box_global(orig)
    except Exception as e:
        print(f"  ❌ 실패: {e}")
        failed_ids.append(gid)
        continue

    all_pbp.append((gid, pbp_df))
    all_inning_rows.append(inning_df)
    all_batter_box.append(batter_df)
    all_pitcher_box.append(pitcher_df)
    all_diag[gid] = diag

print("\nPBP 저장 중...")
save_pbp_games(all_pbp)

inning_summary_df = pd.concat(all_inning_rows, ignore_index=True)
inning_summary_df = save_inning_summary_global(inning_summary_df, YEAR)
print(f"\ninning_summary_global_{YEAR}.xlsx 갱신 완료 "
      f"(누적 {inning_summary_df['game_id'].nunique()}경기, {len(inning_summary_df)}행)")

batter_box_df = pd.concat(all_batter_box, ignore_index=True)
batter_box_df = save_batter_box_global(batter_box_df, YEAR)
print(f"batter_box_global_{YEAR}.xlsx 갱신 완료 (누적 {batter_box_df['game_id'].nunique()}경기, {len(batter_box_df)}행)")

pitcher_box_df = pd.concat(all_pitcher_box, ignore_index=True)
pitcher_box_df = save_pitcher_box_global(pitcher_box_df, YEAR)
print(f"pitcher_box_global_{YEAR}.xlsx 갱신 완료 (누적 {pitcher_box_df['game_id'].nunique()}경기, {len(pitcher_box_df)}행)")

if failed_ids:
    print(f"\n⚠ 처리 실패 {len(failed_ids)}경기: {failed_ids}")

# ── 전체 요약 ──
_tot = lambda k: sum(d[k] for d in all_diag.values())
print(f"\n[요약] 타석 {_tot('pa_rows')}행 / 주루이벤트 {_tot('event_rows')}행 / "
      f"주자아웃 분리 {_tot('split_runner_outs')}행 / 대주자 {_tot('pinch_runner_rows')}행 / "
      f"승부치기 half {_tot('tiebreak_halves')}개")
_stale_b = [(g, x) for g, d in all_diag.items() for x in d['stale_batterid']]
if _stale_b:
    print(f"[요약] WBSC batterid stale {len(_stale_b)}타석 (lineupslot 기준으로 교정됨):")
    for g, (inn, hf, was, now) in _stale_b:
        print(f"   game {g} {inn}회 {hf}: {was} -> {now}")
if _tot('stale_pitcherid'):
    print(f"[요약] WBSC pitcherid stale {_tot('stale_pitcherid')}타석 "
          f"(Pitching Change 추적으로 교정됨)")

# result 번역이 med/low/none 인 행만 따로 검수
# (실제 대조 검수 — PBP vs 공식 boxScore — 는 "WBSC 검수.ipynb"에서 한다.)
review = pd.concat([df for _, df in all_pbp], ignore_index=True)
review = review[review["result_confidence"] != "high"]
print(f"\n검수 필요(번역 신뢰도 high 아님) {len(review)}행:")
review[["game_id", "team", "inning", "topbot", "result", "result_en", "result_confidence"]]
